# DAPI nuclei with configurable perinuclear markers — Cellpose-SAM 3D v5.5

This workflow segments DAPI nuclei in 3D and then classifies them using one or more independently measured perinuclear markers. MAP2, VGLUT, or another configured channel can be used without hard-coded marker names.

## Main capabilities

- Segments DAPI alone with Cellpose-SAM.
- Measures any number of configured image channels inside each nucleus.
- Measures one or more markers in physical 3D shells outside each nucleus.
- Keeps marker intensities separate; it does not add MAP2 and VGLUT values together.
- Combines multiple perinuclear-marker rules with `any` (OR) or `all` (AND).
- Provides one live minimum-positive-fraction slider per enabled marker.
- Updates a live perinuclear-shell layer for each marker as nuclei are filtered.
- Applies morphology, intensity, and physical Z-surface filters in Napari.
- Exports raw nuclei, per-marker nuclei, combined marker-associated nuclei, final filtered nuclei, tables, figures, configuration, and a PDF report.
- Supports TIFF axis standardization, physical calibration, OME-Zarr caching, and resume mode.

The analysis remains configuration-driven so it can later become a standalone application.


**v5.5:** calibrated MNTB ROI, reviewed live density, ROI export/resume. Density results are in `analysis_summary.csv` and separate `mntb_density_summary.csv/.txt`; the original PDF retains its existing content.


Synchronized release: M3 scientific settings, calculations and wording are shared with Windows. Device selection is automatic (CUDA / Apple MPS / CPU). Nominal thickness is preserved in exported configuration.


## How to use

1. Edit the **User configuration** cell and confirm the channel names and zero-based indices.
2. Keep `SEGMENTATION_CHANNEL = "DAPI"` for this nucleus-first workflow.
3. In `PERINUCLEAR_MARKERS`, enable MAP2, VGLUT, both, or another channel listed in `CHANNELS`.
4. Set `PERINUCLEAR_COMBINATION = "any"` to accept nuclei passing at least one marker rule, or `"all"` to require every enabled marker.
5. Run the notebook from top to bottom and inspect each marker shell and associated-nuclei layer in Napari.
6. Adjust the live minimum-positive-fraction sliders and export after the desired final masks are visible.

The example channel panel is DAPI = 0, CC3 = 1, VGLUT = 2, and MAP2 = 3. Change these indices to match the TIFF.

Each marker may use its own inner distance, outer distance, pixel-intensity threshold, and minimum positive-shell fraction. `pixel_threshold = None` uses Otsu as a starting estimate. For final biological comparisons, use thresholds supported by negative controls or a validated background rule.

### Add density to an existing analysis
Set `RUN_MODE = "density_only"`, then run the notebook in order. Choose the previous result folder and its original cropped TIFF. Saved retained nuclei and filter settings are reused; Cellpose, morphology and perinuclear measurements are skipped. Review the ROI in Napari, click **Accept reviewed ROI and calculate density**, then **Save density addendum**. A timestamped subfolder records density, ROI, per-Z volumes, original settings and per-nucleus membership without modifying the original results. Counts outside the selected ROI/Z interval are reported separately.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import re
import textwrap
from datetime import datetime
from pathlib import Path
from time import perf_counter
from xml.etree import ElementTree as ET

import matplotlib.pyplot as plt
import napari
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
import skimage as ski
import tifffile
import torch
from cellpose import io, models
from IPython.display import display
from magicgui.widgets import (
    ComboBox,
    Container,
    FloatSlider,
    Label,
    PushButton,
    Slider,
)
from matplotlib.backends.backend_pdf import PdfPages
from qtpy.QtWidgets import QApplication, QFileDialog
from skimage.measure import marching_cubes, mesh_surface_area, regionprops

In [ ]:
%gui qt

print("Qt event loop integrated with Jupyter.")

## 1. User configuration

This should be the only cell that most users need to edit.

Each channel has:

- `name`: marker name used in Napari, CSV columns, and reports.
- `index`: zero-based position in the TIFF.
- `role`: descriptive metadata (`segmentation`, `measurement`, or `reference`).
- `measure`: whether per-nucleus intensities should be calculated.
- `filter_enabled`: whether the live panel should create min/max controls for that channel.
- `filter_min` and `filter_max`: starting values; use `None` for the observed minimum/maximum.
- `colormap`: Napari colormap name.

In [ ]:
# ----------------------- RUN MODE -----------------------
# "segment": run Cellpose on a newly selected TIFF.
# "density_only": annotate an exported analysis without rerunning measurements.
# "resume": load cellpose_raw_masks.tif and settings from a previous v5 result.
RUN_MODE = "resume"


# ---------------------- TIFF LAYOUT ---------------------
# Usually leave this as None and use the axes stored in the TIFF.
# If a TIFF reports ambiguous axes, enter a string matching its dimensions,
# for example "ZCYX" or "CZYX".
AXES_OVERRIDE = None

# Used only when the TIFF contains more than one time point.
TIME_INDEX = 0

# Manual fallback in (Z, Y, X) µm. Leave as None to use TIFF metadata.
# Example: VOXEL_SPACING_OVERRIDE_UM = (0.68, 0.28, 0.28)
VOXEL_SPACING_OVERRIDE_UM = None

# Correction to nominal post-fixation section thickness.
NOMINAL_SECTION_THICKNESS_UM = 60.0

# ---------------- OPTIONAL OME-ZARR CACHE --------------
# False: read the TIFF normally.
# True: create the cache once, then reuse it on later runs.
# This can improve repeated loading and large-volume access, but it does not
# accelerate Cellpose neural-network inference or 3D mask reconstruction.
USE_OME_ZARR_CACHE = False

# None writes the cache beside the TIFF. For TIFFs on V:/Z: or another network
# drive, a local SSD folder is usually faster, for example:
# OME_ZARR_CACHE_DIRECTORY = Path(r"C:\Cellpose_OME_Zarr_cache")
OME_ZARR_CACHE_DIRECTORY = None

# Chunk order is C, Z, Y, X. Channel chunks of 1 allow efficient loading of a
# selected segmentation channel. Values larger than the image are reduced
# automatically. These defaults are a reasonable compromise for 3D images.
OME_ZARR_CHUNKS_CZYX = (1, 8, 256, 256)

# ---------------------- CHANNEL PANEL -------------------
CHANNELS = [
    {
        "name": "DAPI",
        "index": 0,
        "role": "segmentation",
        "measure": True,
        "filter_enabled": True,
        "filter_min": None,
        "filter_max": None,
        "colormap": "cyan",
    },
    {
        "name": "CC3",
        "index": 1,
        "role": "measurement",
        "measure": True,
        "filter_enabled": True,
        "filter_min": None,
        "filter_max": None,
        "colormap": "green",
    },
    {
        "name": "VGLUT",
        "index": 2,
        "role": "measurement",
        "measure": True,
        "filter_enabled": True,
        "filter_min": None,
        "filter_max": None,
        "colormap": "red",
    },
    {
        "name": "MAP2",
        "index": 3,
        "role": "measurement",
        "measure": True,
        "filter_enabled": True,
        "filter_min": None,
        "filter_max": None,
        "colormap": "magenta",
    },
]

SEGMENTATION_CHANNEL = "DAPI"



# ----------- CONFIGURABLE PERINUCLEAR MARKERS -----------
# Marker intensities are measured separately. Enable one marker or several.
# The example VGLUT entry is disabled initially; change it to True to use it.
PERINUCLEAR_MARKERS = [
    {
        "name": "MAP2",
        "enabled": True,
        "ring_inner_um": 0.0,
        "ring_outer_um": 1.0,
        "pixel_threshold": None,
        "initial_min_positive_fraction": 0.1,
    },
    {
        "name": "VGLUT",
        "enabled": True,
        "ring_inner_um": 0.0,
        "ring_outer_um": 2.0,
        "pixel_threshold": None,
        "initial_min_positive_fraction": 0.10,
    },
]

# "any": a nucleus may pass MAP2 OR VGLUT.
# "all": a nucleus must pass MAP2 AND VGLUT.
PERINUCLEAR_COMBINATION = "all"


# --------------------- CELLPPOSE-SAM --------------------
MODEL_NAME = "cpsam_v2"
CELLPROB_THRESHOLD = -1.0
CELLPOSE_MIN_SIZE_VOXELS = 100
BATCH_SIZE = 32

# Retains the smoothing used in nuclei_seg_v4. Set to 0 or None to disable.
FLOW3D_SMOOTH = [1, 0, 0]

# Optional Cellpose diameter. None lets the model choose its normal behavior.
DIAMETER_PIXELS = None


# -------------------- INITIAL FILTERS -------------------
INITIAL_MIN_VOLUME_UM3 = None  # None = 500 voxels converted to µm³
INITIAL_MIN_SPHERICITY = 0.45
INITIAL_FIRST_TISSUE_Z = 0
INITIAL_LAST_TISSUE_Z = None  # None = final acquired Z plane
INITIAL_Z_GUARD_UM = 0.0
INITIAL_Z_GUARD_MODE = "first"  # none, first, last, or both

# "all": every enabled intensity channel must pass.
# "any": at least one enabled intensity channel must pass.
FILTER_COMBINATION = "all"


# ---------------- INTENSITY MEASUREMENTS ----------------
# Raw measurements are always preserved.
# "raw" does not use background for filtering.
# "global_background_subtracted" estimates background outside all masks
# and uses background-corrected mean intensity for filtering.
INTENSITY_MODE = "raw"
GLOBAL_BACKGROUND_PERCENTILE = 50.0


# ------------------------ OUTPUT ------------------------
# None uses a sanitized form of the complete TIFF stem.
CUSTOM_SAMPLE_NAME = None
COMPUTE_INPUT_SHA256 = True
PDF_AUTHOR = "Nikollas"
# MNTB volume: "constant_xy" ONLY for one Fiji contour used throughout Z.
MNTB_ROI_MODE = "per_slice"
# Optional binary TIFF in the same ZYX grid; preferred when the Fiji ROI is available.
MNTB_ROI_MASK_PATH = None


## 2. Reusable core functions

These functions contain the analysis logic that can later be moved into the standalone app.

In [ ]:
UNIT_TO_UM = {
    "µm": 1.0,
    "um": 1.0,
    "micron": 1.0,
    "microns": 1.0,
    "micrometer": 1.0,
    "micrometers": 1.0,
    "nm": 1e-3,
    "nanometer": 1e-3,
    "nanometers": 1e-3,
    "mm": 1e3,
    "millimeter": 1e3,
    "millimeters": 1e3,
    "cm": 1e4,
    "centimeter": 1e4,
    "centimeters": 1e4,
    "m": 1e6,
    "meter": 1e6,
    "meters": 1e6,
}


def safe_key(value: str) -> str:
    key = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")
    if not key:
        raise ValueError(f"Cannot create a safe name from {value!r}.")
    return key


def safe_filename(value: str, maximum_length: int = 100) -> str:
    name = re.sub(r'[<>:"/\\|?*]+', "_", str(value)).strip(" ._")
    name = re.sub(r"\s+", "_", name)
    return (name or "sample")[:maximum_length]


def validate_channel_configuration(channels, segmentation_channel):
    if not channels:
        raise ValueError("CHANNELS cannot be empty.")

    names = [str(item["name"]) for item in channels]
    indices = [int(item["index"]) for item in channels]
    keys = [safe_key(name).lower() for name in names]

    if len(names) != len(set(names)):
        raise ValueError("Every channel name must be unique.")
    if len(indices) != len(set(indices)):
        raise ValueError("Every configured channel index must be unique.")
    if len(keys) != len(set(keys)):
        raise ValueError(
            "Channel names must remain unique after conversion to CSV-safe names."
        )
    if any(index < 0 for index in indices):
        raise ValueError("Channel indices cannot be negative.")
    if segmentation_channel not in names:
        raise ValueError(
            f"SEGMENTATION_CHANNEL {segmentation_channel!r} is not in CHANNELS."
        )

    valid_roles = {"segmentation", "measurement", "reference"}
    invalid_roles = [
        item.get("role")
        for item in channels
        if item.get("role") not in valid_roles
    ]
    if invalid_roles:
        raise ValueError(
            f"Invalid channel roles: {invalid_roles}. Use {sorted(valid_roles)}."
        )


def unit_factor_to_um(unit):
    key = str(unit or "").strip().replace("μ", "µ").lower()
    if key not in UNIT_TO_UM:
        raise ValueError(f"Unsupported or missing physical unit: {unit!r}")
    return UNIT_TO_UM[key]


def resolution_as_float(value):
    if isinstance(value, (tuple, list)) and len(value) == 2:
        return float(value[0]) / float(value[1])
    return float(value)


def ome_voxel_spacing_um(ome_xml):
    if not ome_xml:
        return None

    root = ET.fromstring(ome_xml)
    pixels = next(
        (
            element
            for element in root.iter()
            if element.tag.rsplit("}", 1)[-1] == "Pixels"
        ),
        None,
    )
    if pixels is None:
        return None

    spacing = {}
    for axis in ("X", "Y", "Z"):
        value = pixels.attrib.get(f"PhysicalSize{axis}")
        if value is None:
            return None
        unit = pixels.attrib.get(f"PhysicalSize{axis}Unit", "µm")
        spacing[axis] = float(value) * unit_factor_to_um(unit)

    return spacing["Z"], spacing["Y"], spacing["X"]


def imagej_voxel_spacing_um(tif):
    metadata = tif.imagej_metadata or {}
    if "spacing" not in metadata or "unit" not in metadata:
        return None

    unit = metadata["unit"]
    z_um = float(metadata["spacing"]) * unit_factor_to_um(unit)
    tags = tif.pages[0].tags
    x_tag = tags.get("XResolution")
    y_tag = tags.get("YResolution")
    if x_tag is None or y_tag is None:
        return None

    x_pixels_per_unit = resolution_as_float(x_tag.value)
    y_pixels_per_unit = resolution_as_float(y_tag.value)
    resolution_unit_tag = tags.get("ResolutionUnit")
    resolution_unit = int(resolution_unit_tag.value) if resolution_unit_tag else 1

    if resolution_unit == 2:
        x_um = 25400.0 / x_pixels_per_unit
        y_um = 25400.0 / y_pixels_per_unit
    elif resolution_unit == 3:
        x_um = 10000.0 / x_pixels_per_unit
        y_um = 10000.0 / y_pixels_per_unit
    else:
        factor = unit_factor_to_um(unit)
        x_um = factor / x_pixels_per_unit
        y_um = factor / y_pixels_per_unit

    return z_um, y_um, x_um


def read_voxel_spacing_um(path, manual_override=None):
    with tifffile.TiffFile(path) as tif:
        spacing = ome_voxel_spacing_um(tif.ome_metadata)
        source = "OME-XML"
        if spacing is None:
            spacing = imagej_voxel_spacing_um(tif)
            source = "ImageJ/Fiji TIFF metadata"

    if spacing is None:
        if manual_override is None:
            raise ValueError(
                "Complete physical Z/Y/X spacing was not found. Set "
                "VOXEL_SPACING_OVERRIDE_UM = (Z, Y, X) in the configuration."
            )
        spacing = manual_override
        source = "manual configuration"

    spacing = tuple(float(value) for value in spacing)
    if len(spacing) != 3:
        raise ValueError("Voxel spacing must contain exactly (Z, Y, X).")
    if not all(np.isfinite(value) and value > 0 for value in spacing):
        raise ValueError(f"Invalid physical voxel spacing: {spacing}")
    return spacing, source


def standardize_to_czyx(data, axes, axes_override=None, time_index=0):
    axes = str(axes_override or axes).upper()
    if len(axes) != data.ndim:
        raise ValueError(
            f"Axes {axes!r} has {len(axes)} characters but data has "
            f"{data.ndim} dimensions."
        )
    if len(set(axes)) != len(axes):
        raise ValueError(f"Repeated axes are not supported: {axes!r}")

    if "T" in axes:
        t_axis = axes.index("T")
        if not 0 <= time_index < data.shape[t_axis]:
            raise ValueError(
                f"TIME_INDEX {time_index} is outside 0..{data.shape[t_axis] - 1}."
            )
        data = np.take(data, time_index, axis=t_axis)
        axes = axes.replace("T", "")

    for axis_name in list(axes):
        if axis_name not in "CZYX":
            axis_position = axes.index(axis_name)
            if data.shape[axis_position] != 1:
                raise ValueError(
                    f"Unsupported non-singleton axis {axis_name!r} in {axes!r}. "
                    "Set AXES_OVERRIDE explicitly if this axis represents Z or C."
                )
            data = np.take(data, 0, axis=axis_position)
            axes = axes.replace(axis_name, "")

    if "Y" not in axes or "X" not in axes:
        raise ValueError(f"TIFF axes must include Y and X; received {axes!r}.")

    if "Z" not in axes:
        data = np.expand_dims(data, axis=0)
        axes = "Z" + axes
    if "C" not in axes:
        data = np.expand_dims(data, axis=0)
        axes = "C" + axes

    order = [axes.index(axis) for axis in "CZYX"]
    return np.ascontiguousarray(np.transpose(data, order)), axes


def load_tiff_czyx(path, axes_override=None, time_index=0):
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        reported_shape = tuple(series.shape)
        reported_axes = str(series.axes)
        dtype = str(series.dtype)
        data = series.asarray()

    czyx, axes_used = standardize_to_czyx(
        data,
        reported_axes,
        axes_override=axes_override,
        time_index=time_index,
    )
    information = {
        "reported_shape": reported_shape,
        "reported_axes": reported_axes,
        "axes_interpreted_before_reorder": axes_used,
        "standardized_axes": "CZYX",
        "standardized_shape": tuple(czyx.shape),
        "dtype": dtype,
    }
    return czyx, information



def source_file_signature(path):
    path = Path(path)
    stat = path.stat()
    return {
        "filename": path.name,
        "size_bytes": int(stat.st_size),
        "modified_time_ns": int(stat.st_mtime_ns),
    }


def require_ome_zarr_packages():
    try:
        import zarr
        from numcodecs import Blosc
    except ImportError as error:
        raise ImportError(
            "OME-Zarr caching is enabled, but the optional packages are not "
            "installed. In the cellposesam environment run: pip install "
            "\"zarr<3\" numcodecs, restart the kernel, and rerun the notebook."
        ) from error

    major_version = int(str(zarr.__version__).split(".", 1)[0])
    if major_version >= 3:
        raise RuntimeError(
            "This notebook currently uses the stable Zarr v2 OME-NGFF writer. "
            "Install a compatible version with: pip install \"zarr<3\" numcodecs"
        )
    return zarr, Blosc


def normalized_chunks_czyx(requested_chunks, image_shape):
    if len(requested_chunks) != 4:
        raise ValueError("OME_ZARR_CHUNKS_CZYX must contain (C, Z, Y, X).")

    chunks = tuple(int(value) for value in requested_chunks)
    if any(value <= 0 for value in chunks):
        raise ValueError("Every OME-Zarr chunk dimension must be positive.")
    return tuple(min(chunk, int(size)) for chunk, size in zip(chunks, image_shape))


def ome_zarr_cache_path(tiff_path, cache_directory=None):
    tiff_path = Path(tiff_path)
    if cache_directory is None:
        return tiff_path.parent / f"{tiff_path.stem}.ome.zarr"

    # A local cache folder may receive identically named TIFFs from different
    # source folders. Add a short path digest to prevent those collisions.
    parent = Path(cache_directory)
    path_digest = hashlib.sha256(
        str(tiff_path.absolute()).encode("utf-8")
    ).hexdigest()[:12]
    return parent / f"{tiff_path.stem}__{path_digest}.ome.zarr"


def write_single_scale_ome_zarr(
    image_czyx,
    output_path,
    voxel_spacing_zyx_um,
    channel_configuration,
    source_tiff,
    source_information,
    requested_chunks=(1, 8, 256, 256),
):
    zarr, Blosc = require_ome_zarr_packages()
    output_path = Path(output_path)
    if output_path.exists():
        raise FileExistsError(
            f"OME-Zarr target already exists and will not be overwritten: {output_path}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    chunks = normalized_chunks_czyx(requested_chunks, image_czyx.shape)
    compressor = Blosc(cname="zstd", clevel=3, shuffle=Blosc.BITSHUFFLE)

    root = zarr.open_group(str(output_path), mode="w")
    level_zero = root.create_dataset(
        "0",
        shape=image_czyx.shape,
        chunks=chunks,
        dtype=image_czyx.dtype,
        compressor=compressor,
        overwrite=False,
    )
    level_zero[:] = image_czyx

    z_um, y_um, x_um = (float(value) for value in voxel_spacing_zyx_um)
    root.attrs["multiscales"] = [
        {
            "version": "0.4",
            "name": Path(source_tiff).stem,
            "axes": [
                {"name": "c", "type": "channel"},
                {"name": "z", "type": "space", "unit": "micrometer"},
                {"name": "y", "type": "space", "unit": "micrometer"},
                {"name": "x", "type": "space", "unit": "micrometer"},
            ],
            "datasets": [
                {
                    "path": "0",
                    "coordinateTransformations": [
                        {"type": "scale", "scale": [1.0, z_um, y_um, x_um]}
                    ],
                }
            ],
            "type": "image",
        }
    ]
    root.attrs["nuclei_segmentation_cache"] = {
        "schema_version": 1,
        "standardized_axes": "CZYX",
        "source_signature": source_file_signature(source_tiff),
        "source_image_information": {
            key: list(value) if isinstance(value, tuple) else value
            for key, value in source_information.items()
        },
        "voxel_spacing_zyx_um": [z_um, y_um, x_um],
        "chunks_czyx": list(chunks),
        "configured_channels": [
            {
                "name": str(item["name"]),
                "index": int(item["index"]),
            }
            for item in channel_configuration
        ],
    }
    return chunks


def load_tiff_with_optional_ome_zarr_cache(
    tiff_path,
    voxel_spacing_zyx_um,
    channel_configuration,
    use_cache=False,
    cache_directory=None,
    requested_chunks=(1, 8, 256, 256),
    axes_override=None,
    time_index=0,
):
    tiff_path = Path(tiff_path)
    cache_path = ome_zarr_cache_path(tiff_path, cache_directory)
    cache_record = {
        "enabled": bool(use_cache),
        "path_at_analysis": str(cache_path) if use_cache else None,
        "reused_existing_cache": False,
        "storage_backend": "TIFF",
        "tiff_load_seconds": None,
        "ome_zarr_load_seconds": None,
        "ome_zarr_conversion_seconds": None,
        "chunks_czyx": None,
    }

    if use_cache and cache_path.exists():
        zarr, _ = require_ome_zarr_packages()
        load_start = perf_counter()
        root = zarr.open_group(str(cache_path), mode="r")
        cache_metadata = dict(root.attrs.get("nuclei_segmentation_cache", {}))
        expected_signature = cache_metadata.get("source_signature")
        observed_signature = source_file_signature(tiff_path)

        if expected_signature != observed_signature:
            raise ValueError(
                "The existing OME-Zarr cache does not match the TIFF size and "
                "modification time. Rename or remove the stale cache, or choose "
                "a different OME_ZARR_CACHE_DIRECTORY before continuing.\n"
                f"Cache: {cache_path}"
            )
        if cache_metadata.get("standardized_axes") != "CZYX" or "0" not in root:
            raise ValueError(
                f"The selected cache is not a compatible CZYX OME-Zarr store: {cache_path}"
            )

        image_czyx = np.asarray(root["0"])
        source_information = dict(cache_metadata.get("source_image_information", {}))
        for key in ("reported_shape", "standardized_shape"):
            if key in source_information:
                source_information[key] = tuple(source_information[key])

        elapsed = perf_counter() - load_start
        cache_record.update(
            {
                "reused_existing_cache": True,
                "storage_backend": "OME-Zarr cache",
                "ome_zarr_load_seconds": float(elapsed),
                "chunks_czyx": list(root["0"].chunks),
            }
        )
        return image_czyx, source_information, cache_record

    tiff_start = perf_counter()
    image_czyx, source_information = load_tiff_czyx(
        tiff_path,
        axes_override=axes_override,
        time_index=time_index,
    )
    cache_record["tiff_load_seconds"] = float(perf_counter() - tiff_start)

    if use_cache:
        conversion_start = perf_counter()
        chunks = write_single_scale_ome_zarr(
            image_czyx=image_czyx,
            output_path=cache_path,
            voxel_spacing_zyx_um=voxel_spacing_zyx_um,
            channel_configuration=channel_configuration,
            source_tiff=tiff_path,
            source_information=source_information,
            requested_chunks=requested_chunks,
        )
        cache_record.update(
            {
                "storage_backend": "TIFF (OME-Zarr cache created for later runs)",
                "ome_zarr_conversion_seconds": float(perf_counter() - conversion_start),
                "chunks_czyx": list(chunks),
            }
        )

    return image_czyx, source_information, cache_record


def file_sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def software_versions():
    def package_version(name):
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return "unknown"

    return {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "tifffile": tifffile.__version__,
        "scikit_image": ski.__version__,
        "torch": torch.__version__,
        "cellpose": package_version("cellpose"),
        "napari": napari.__version__,
        "magicgui": package_version("magicgui"),
        "zarr": package_version("zarr"),
        "numcodecs": package_version("numcodecs"),
    }


def calculate_surface_metrics(binary_mask, voxel_spacing):
    volume_um3 = float(binary_mask.sum() * np.prod(voxel_spacing))
    try:
        padded = np.pad(binary_mask.astype(np.uint8), 1)
        vertices, faces, _, _ = marching_cubes(
            padded,
            level=0.5,
            spacing=voxel_spacing,
        )
        surface_area_um2 = float(mesh_surface_area(vertices, faces))
        sphericity = float(
            np.pi ** (1 / 3)
            * (6 * volume_um3) ** (2 / 3)
            / surface_area_um2
        )
    except (ValueError, RuntimeError):
        surface_area_um2 = np.nan
        sphericity = np.nan
    return surface_area_um2, sphericity


def calculate_mask_properties(
    masks,
    channel_volumes,
    channel_configuration,
    voxel_spacing,
    background_percentile=50.0,
):
    voxel_volume_um3 = float(np.prod(voxel_spacing))
    rows = []
    measured_channels = [
        item for item in channel_configuration if item.get("measure", True)
    ]

    outside_masks = masks == 0
    background_by_channel = {}
    for item in measured_channels:
        name = item["name"]
        values = channel_volumes[name][outside_masks]
        if values.size == 0:
            values = channel_volumes[name].ravel()
        background_by_channel[name] = float(
            np.percentile(values, background_percentile)
        )

    for region in regionprops(masks):
        min_z, min_y, min_x, max_z_exclusive, max_y_exclusive, max_x_exclusive = (
            region.bbox
        )
        max_z = int(max_z_exclusive - 1)
        max_y = int(max_y_exclusive - 1)
        max_x = int(max_x_exclusive - 1)
        volume_um3 = float(region.area * voxel_volume_um3)
        surface_area_um2, sphericity = calculate_surface_metrics(
            region.image,
            voxel_spacing,
        )
        equivalent_diameter_um = float((6 * volume_um3 / np.pi) ** (1 / 3))
        centroid_z, centroid_y, centroid_x = region.centroid

        row = {
            "label": int(region.label),
            "size_voxels": int(region.area),
            "volume_um3": volume_um3,
            "surface_area_um2": surface_area_um2,
            "sphericity": sphericity,
            "equivalent_sphere_diameter_um": equivalent_diameter_um,
            "centroid_z_px": float(centroid_z),
            "centroid_y_px": float(centroid_y),
            "centroid_x_px": float(centroid_x),
            "centroid_z_um": float(centroid_z * voxel_spacing[0]),
            "centroid_y_um": float(centroid_y * voxel_spacing[1]),
            "centroid_x_um": float(centroid_x * voxel_spacing[2]),
            "min_z": int(min_z),
            "max_z": max_z,
            "min_y": int(min_y),
            "max_y": max_y,
            "min_x": int(min_x),
            "max_x": max_x,
            "z_extent_planes": int(max_z_exclusive - min_z),
            "z_extent_um": float((max_z_exclusive - min_z) * voxel_spacing[0]),
        }

        for item in measured_channels:
            name = item["name"]
            key = safe_key(name)
            local_volume = channel_volumes[name][region.slice]
            values = local_volume[region.image].astype(np.float64, copy=False)
            mean_value = float(values.mean())
            background = background_by_channel[name]

            row[f"mean_intensity_{key}"] = mean_value
            row[f"median_intensity_{key}"] = float(np.median(values))
            row[f"max_intensity_{key}"] = float(values.max())
            row[f"p95_intensity_{key}"] = float(np.percentile(values, 95))
            row[f"integrated_intensity_{key}"] = float(values.sum())
            row[f"background_corrected_mean_{key}"] = float(
                mean_value - background
            )

        rows.append(row)

    if not rows:
        raise RuntimeError("Cellpose did not produce any masks to measure.")

    table = pd.DataFrame(rows).set_index("label").sort_index()
    table.index = table.index.astype(int)
    table.index.name = "label"
    background_table = pd.DataFrame(
        [
            {
                "channel": name,
                "background_percentile": float(background_percentile),
                "background_intensity": value,
            }
            for name, value in background_by_channel.items()
        ]
    )
    return table, background_table




def resolve_marker_threshold(intensity_volume, configured_threshold=None):
    if configured_threshold is not None:
        threshold = float(configured_threshold)
        if not np.isfinite(threshold):
            raise ValueError("The configured marker threshold must be finite.")
        return threshold, "manual configuration"

    finite_values = np.asarray(intensity_volume)[
        np.isfinite(intensity_volume)
    ].ravel()
    if finite_values.size == 0:
        raise ValueError("The marker volume contains no finite intensity values.")

    # Cap the deterministic sample to keep threshold estimation inexpensive.
    stride = max(1, int(np.ceil(finite_values.size / 2_000_000)))
    sample = finite_values[::stride]
    if float(sample.min()) == float(sample.max()):
        return float(sample.min()), "constant image"

    return float(ski.filters.threshold_otsu(sample)), "Otsu starting estimate"


def calculate_perinuclear_marker_metrics(
    nucleus_masks,
    intensity_volume,
    voxel_spacing,
    channel_name,
    inner_distance_um=0.5,
    outer_distance_um=3.0,
    pixel_threshold=None,
):
    nucleus_masks = np.asarray(nucleus_masks)
    intensity_volume = np.asarray(intensity_volume)
    voxel_spacing = tuple(float(value) for value in voxel_spacing)
    inner_distance_um = float(inner_distance_um)
    outer_distance_um = float(outer_distance_um)

    if nucleus_masks.ndim != 3:
        raise ValueError("Perinuclear analysis requires a 3D ZYX label image.")
    if intensity_volume.shape != nucleus_masks.shape:
        raise ValueError(
            "The marker volume and DAPI masks must have identical ZYX shapes."
        )
    if len(voxel_spacing) != 3 or any(value <= 0 for value in voxel_spacing):
        raise ValueError("voxel_spacing must contain positive (Z, Y, X) values.")
    if inner_distance_um < 0 or outer_distance_um <= inner_distance_um:
        raise ValueError(
            "Perinuclear ring distances must satisfy 0 <= inner < outer."
        )

    resolved_threshold, threshold_source = resolve_marker_threshold(
        intensity_volume,
        configured_threshold=pixel_threshold,
    )
    prefix = safe_key(channel_name).lower()
    ring_labels = np.zeros_like(nucleus_masks)
    pad_zyx = tuple(
        int(np.ceil(outer_distance_um / spacing))
        for spacing in voxel_spacing
    )

    rows = []
    for region in regionprops(nucleus_masks):
        min_z, min_y, min_x, max_z, max_y, max_x = region.bbox
        starts = (
            max(0, min_z - pad_zyx[0]),
            max(0, min_y - pad_zyx[1]),
            max(0, min_x - pad_zyx[2]),
        )
        stops = (
            min(nucleus_masks.shape[0], max_z + pad_zyx[0]),
            min(nucleus_masks.shape[1], max_y + pad_zyx[1]),
            min(nucleus_masks.shape[2], max_x + pad_zyx[2]),
        )
        local_slice = tuple(
            slice(start, stop) for start, stop in zip(starts, stops)
        )
        local_labels = nucleus_masks[local_slice]
        local_nucleus = local_labels == int(region.label)

        distance_um = ndi.distance_transform_edt(
            ~local_nucleus,
            sampling=voxel_spacing,
        )
        local_ring = (
            (distance_um > inner_distance_um)
            & (distance_um <= outer_distance_um)
            & (local_labels == 0)
        )
        values = intensity_volume[local_slice][local_ring].astype(
            np.float64,
            copy=False,
        )

        local_ring_labels = ring_labels[local_slice]
        display_ring = local_ring & (local_ring_labels == 0)
        local_ring_labels[display_ring] = int(region.label)

        if values.size == 0:
            mean_value = np.nan
            median_value = np.nan
            p75_value = np.nan
            positive_fraction = 0.0
        else:
            mean_value = float(values.mean())
            median_value = float(np.median(values))
            p75_value = float(np.percentile(values, 75))
            positive_fraction = float(np.mean(values >= resolved_threshold))

        rows.append(
            {
                "label": int(region.label),
                f"{prefix}_ring_voxels": int(values.size),
                f"{prefix}_ring_mean": mean_value,
                f"{prefix}_ring_median": median_value,
                f"{prefix}_ring_p75": p75_value,
                f"{prefix}_ring_positive_fraction": positive_fraction,
            }
        )

    if not rows:
        raise RuntimeError("No DAPI masks were available for MAP2 association.")

    table = pd.DataFrame(rows).set_index("label").sort_index()
    table.index = table.index.astype(int)
    table.index.name = "label"
    return table, ring_labels, resolved_threshold, threshold_source


def calculate_z_guard_properties(
    properties,
    first_tissue_z,
    last_tissue_z,
    guard_um,
    guard_mode,
    z_spacing_um,
    max_z_index,
):
    first_tissue_z = int(first_tissue_z)
    last_tissue_z = int(last_tissue_z)
    guard_um = float(guard_um)
    guard_mode = str(guard_mode).lower()

    if not 0 <= first_tissue_z <= last_tissue_z <= max_z_index:
        raise ValueError(
            f"Tissue Z planes must satisfy 0 <= first <= last <= {max_z_index}."
        )
    if guard_um < 0:
        raise ValueError("Z guard thickness cannot be negative.")
    if guard_mode not in {"none", "first", "last", "both"}:
        raise ValueError("Z guard mode must be none, first, last, or both.")

    min_z = properties["min_z"].astype(int)
    max_z = properties["max_z"].astype(int)
    distance_first = ((min_z - first_tissue_z) * z_spacing_um).clip(lower=0)
    distance_last = ((last_tissue_z - max_z) * z_spacing_um).clip(lower=0)

    excluded_first = (min_z < first_tissue_z) | (distance_first <= guard_um)
    excluded_last = (max_z > last_tissue_z) | (distance_last <= guard_um)

    if guard_mode == "none":
        excluded = pd.Series(False, index=properties.index)
    elif guard_mode == "first":
        excluded = excluded_first
    elif guard_mode == "last":
        excluded = excluded_last
    else:
        excluded = excluded_first | excluded_last

    return pd.DataFrame(
        {
            "distance_from_first_z_surface_um": distance_first,
            "distance_from_last_z_surface_um": distance_last,
            "distance_from_nearest_z_surface_um": np.minimum(
                distance_first,
                distance_last,
            ),
            "excluded_near_first_z_surface": excluded_first.astype(bool),
            "excluded_near_last_z_surface": excluded_last.astype(bool),
            "excluded_by_z_guard": excluded.astype(bool),
        },
        index=properties.index,
    )


def calibrated_label_tiff(path, labels, voxel_spacing, require_uint16=False):
    maximum_label = int(labels.max())
    if maximum_label <= np.iinfo(np.uint16).max:
        output_dtype = np.uint16
    elif require_uint16:
        raise ValueError(
            "The mask contains more than 65,535 label IDs and cannot be "
            "saved as a uint16 SyGlass-compatible label TIFF."
        )
    else:
        output_dtype = np.uint32

    tifffile.imwrite(
        path,
        labels.astype(output_dtype, copy=False),
        imagej=True,
        metadata={
            "axes": "ZYX",
            "spacing": float(voxel_spacing[0]),
            "unit": "um",
        },
        resolution=(
            1.0 / float(voxel_spacing[2]),
            1.0 / float(voxel_spacing[1]),
        ),
        photometric="minisblack",
    )

# MNTB sampling volume: original image grid, never Cellpose-resampled data.
def reconstruct_mntb_roi(image_czyx, mode="per_slice"):
    if mode not in {"per_slice", "constant_xy"}:
        raise ValueError("ROI mode must be per_slice or constant_xy.")
    shape = tuple(image_czyx.shape[1:])
    roi = np.zeros(shape, dtype=bool)
    for z in range(shape[0]):
        support = np.zeros(shape[1:], dtype=bool)
        for c in range(image_czyx.shape[0]):
            plane = np.asarray(image_czyx[c, z])
            support |= np.isfinite(plane) & (plane != 0)
        roi[z] = ndi.binary_fill_holes(support)
    if mode == "constant_xy":
        # Use ONLY if the same Fiji contour was applied to every Z plane.
        footprint = ndi.binary_fill_holes(np.any(roi, axis=0))
        roi[:] = footprint
    return roi


def density_z_planes(settings, nz, dz):
    first, last = int(settings["first_tissue_z"]), int(settings["last_tissue_z"])
    guard, mode = float(settings["z_guard_um"]), settings["z_guard_mode"]
    if not 0 <= first <= last < nz or not np.isfinite(guard) or guard < 0:
        raise ValueError("Invalid tissue range or guard thickness.")
    if mode not in {"none", "first", "last", "both"}:
        raise ValueError("Invalid guard mode.")
    z = np.arange(nz)
    tissue = (z >= first) & (z <= last)
    effective = tissue.copy()
    # Match existing whole-object guards: distance <= guard is excluded.
    # Thus even guard=0 excludes an active surface plane.
    if mode in {"first", "both"}:
        effective &= (z - first) * dz > guard
    if mode in {"last", "both"}:
        effective &= (last - z) * dz > guard
    return tissue, effective


def roi_centroid_membership(properties, roi):
    coords = properties[["centroid_z_px", "centroid_y_px", "centroid_x_px"]].to_numpy()
    finite = np.isfinite(coords).all(axis=1)
    indices = np.floor(np.where(np.isfinite(coords), coords, -1) + 0.5).astype(int)
    valid = finite & (indices >= 0).all(axis=1) & (indices < np.array(roi.shape)).all(axis=1)
    inside = np.zeros(len(properties), dtype=bool)
    inside[valid] = roi[tuple(indices[valid].T)]
    return pd.Series(inside, index=properties.index), indices[:, 0]


def calculate_density_summary(
    plane_counts,
    settings,
    spacing,
    retained_count,
    reviewed,
):
    spacing = np.asarray(spacing, dtype=float)

    if (
        spacing.shape != (3,)
        or not np.isfinite(spacing).all()
        or (spacing <= 0).any()
    ):
        raise ValueError(
            "Voxel spacing must contain three positive finite values."
        )

    tissue, effective = density_z_planes(
        settings,
        len(plane_counts),
        spacing[0],
    )

    voxel_volume = float(np.prod(spacing))

    total = int(np.sum(plane_counts))
    tissue_n = int(np.sum(plane_counts[tissue]))
    effective_n = int(np.sum(plane_counts[effective]))

    effective_volume = effective_n * voxel_volume

    density_valid = bool(reviewed) and effective_volume > 0

    density_mm3 = (
        retained_count * 1e9 / effective_volume
        if density_valid
        else None
    )

    density_100um_cube = (
        retained_count * 1e6 / effective_volume
        if density_valid
        else None
    )

    # Optional correction to nominal section thickness.
    nominal = float(NOMINAL_SECTION_THICKNESS_UM)

    # Full tissue thickness BEFORE applying guards.
    # Inclusive plane count, consistent with voxel-based volume.
    first_z = int(settings["first_tissue_z"])
    last_z = int(settings["last_tissue_z"])

    number_of_tissue_planes = last_z - first_z + 1
    measured = float(number_of_tissue_planes * spacing[0])

    if not np.isfinite(nominal) or nominal <= 0:
        raise ValueError(
            "Nominal section thickness must be positive and finite."
        )

    volume_factor = None
    density_factor = None
    corrected_volume = None
    corrected_density = None

    if measured is not None:
        measured = float(measured)

        if not np.isfinite(measured) or measured <= 0:
            raise ValueError(
                "Measured full tissue thickness must be positive and finite."
            )

        volume_factor = nominal / measured
        density_factor = measured / nominal

        if reviewed:
            corrected_volume = effective_volume * volume_factor

        if density_valid:
            corrected_density = density_100um_cube * density_factor

    return {
        "roi_reviewed": bool(reviewed),

        "roi_stack_voxels": total,
        "roi_stack_volume_um3": total * voxel_volume,

        "roi_tissue_voxels": tissue_n,
        "roi_tissue_volume_um3": tissue_n * voxel_volume,

        "roi_effective_voxels": effective_n,
        "roi_effective_volume_um3": effective_volume,
        "roi_effective_volume_mm3": effective_volume / 1e9,

        "retained_nuclei_in_roi": int(retained_count),
        "retained_nuclei_per_mm3": density_mm3,
        "retained_nuclei_per_100um_cube": density_100um_cube,

        "nominal_section_thickness_um": nominal,
        "measured_full_tissue_thickness_um": measured,
        "thickness_volume_correction_factor": volume_factor,
        "thickness_density_correction_factor": density_factor,

        "nominal_corrected_effective_volume_um3": corrected_volume,
        "nominal_corrected_nuclei_per_100um_cube": corrected_density,

        "density_status": (
            "review ROI"
            if not reviewed
            else "empty sampling volume"
            if effective_n == 0
            else "valid"
        ),

        "counting_rule": (
            "filtered nuclei; nearest-voxel centroid in ROI "
            "and effective Z; existing whole-object guards"
        ),

        "interpretation": (
            "Descriptive retained-nucleus density; "
            "not an unbiased stereological estimator."
        ),

        "thickness_correction_assumption": (
            "Uniform Z scaling to nominal post-fixation section "
            "thickness; unchanged XY area; full tissue thickness "
            "measured before guards."
        ),
    }

def format_density_panel(result):
    raw_density = result["retained_nuclei_per_100um_cube"]
    corrected_density = result[
        "nominal_corrected_nuclei_per_100um_cube"
    ]
    corrected_volume = result[
        "nominal_corrected_effective_volume_um3"
    ]
    measured = result["measured_full_tissue_thickness_um"]
    nominal = result["nominal_section_thickness_um"]

    lines = [
        f"Counts (after filtering): {result['retained_nuclei_in_roi']:,}",
        (
            "Tissue volume (after Napari view refinement): "
            f"{result['roi_tissue_volume_um3']:,.1f} µm³"
        ),
        (
            "Tissue volume - Guard: "
            f"{result['roi_effective_volume_um3']:,.1f} µm³"
        ),
    ]

    if raw_density is None:
        lines.append(
            f"Density unavailable: {result['density_status']}"
        )
    else:
        lines.append(
            f"Tissue volume - Guard (density): {raw_density:,.1f} nuclei/(100 µm)³"
        )

    if measured is None:
        lines.append(
            "Thickness correction: full tissue thickness not provided"
        )
    else:
        lines.append(
            f"Scaled to original section thickness: {measured:g} µm → original {nominal:g} µm"
        )

        if corrected_volume is not None:
            lines.append(
                f"Scaled to tissue Volume - Guard: {corrected_volume:,.1f} µm³"
            )

        if corrected_density is not None:
            lines.append(
                f"Scaled to tissue Volume - Guard (density): {corrected_density:,.1f} "
                "nuclei/(100 µm)³"
            )

    return "\n".join(lines)


## 3. Select a TIFF or resume a previous result

Resume mode restores generic channel/filter settings from `analysis_config.json` and loads `cellpose_raw_masks.tif`. It deliberately asks for the original TIFF again so an obsolete stored path does not silently select the wrong image.

In [ ]:
validate_channel_configuration(CHANNELS, SEGMENTATION_CHANNEL)

if RUN_MODE not in {"segment", "resume", "density_only"}:
    raise ValueError('RUN_MODE must be segment, resume, or density_only.')

qt_app = QApplication.instance()
if qt_app is None:
    qt_app = QApplication([])

previous_result_folder = None
previous_config = None

if RUN_MODE in {"resume", "density_only"}:
    selected_folder = QFileDialog.getExistingDirectory(
        None,
        "Select the previous generalized segmentation result folder",
        "",
    )
    if not selected_folder:
        raise RuntimeError("No previous-result folder was selected.")

    previous_result_folder = Path(selected_folder)
    previous_config_path = previous_result_folder / "analysis_config.json"
    previous_masks_path = previous_result_folder / "cellpose_raw_masks.tif"

    missing = [
        path
        for path in (previous_config_path, previous_masks_path)
        if not path.is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "The selected result folder is missing:\n"
            + "\n".join(str(path) for path in missing)
        )

    previous_config = json.loads(previous_config_path.read_text(encoding="utf-8"))
    NOMINAL_SECTION_THICKNESS_UM = float(previous_config.get("nominal_section_thickness_um", NOMINAL_SECTION_THICKNESS_UM))
    CHANNELS = previous_config["channels"]
    SEGMENTATION_CHANNEL = previous_config["segmentation_channel"]
    FILTER_COMBINATION = previous_config["filters"]["intensity_combination"]
    INTENSITY_MODE = previous_config["intensity_measurements"]["mode"]
    GLOBAL_BACKGROUND_PERCENTILE = previous_config[
        "intensity_measurements"
    ]["global_background_percentile"]
    INITIAL_MIN_VOLUME_UM3 = previous_config["filters"]["minimum_volume_um3"]
    INITIAL_MIN_SPHERICITY = previous_config["filters"]["minimum_sphericity"]
    INITIAL_FIRST_TISSUE_Z = previous_config["filters"]["first_tissue_z"]
    INITIAL_LAST_TISSUE_Z = previous_config["filters"]["last_tissue_z"]
    INITIAL_Z_GUARD_UM = previous_config["filters"]["z_guard_um"]
    INITIAL_Z_GUARD_MODE = previous_config["filters"]["z_guard_mode"]
    MODEL_NAME = previous_config["cellpose"]["model_name"]
    CELLPROB_THRESHOLD = previous_config["cellpose"]["cellprob_threshold"]
    CELLPOSE_MIN_SIZE_VOXELS = previous_config["cellpose"][
        "minimum_size_voxels"
    ]
    BATCH_SIZE = previous_config["cellpose"]["batch_size"]
    FLOW3D_SMOOTH = previous_config["cellpose"].get("flow3d_smooth")
    DIAMETER_PIXELS = previous_config["cellpose"].get("diameter_pixels")

    if "perinuclear_markers" in previous_config:
        PERINUCLEAR_MARKERS = []
        for previous_marker in previous_config["perinuclear_markers"]:
            PERINUCLEAR_MARKERS.append(
                {
                    "name": previous_marker["name"],
                    "enabled": previous_marker.get("enabled", True),
                    "ring_inner_um": previous_marker["ring_inner_um"],
                    "ring_outer_um": previous_marker["ring_outer_um"],
                    "pixel_threshold": previous_marker.get(
                        "resolved_pixel_threshold",
                        previous_marker.get("pixel_threshold"),
                    ),
                    "initial_min_positive_fraction": previous_marker.get(
                        "minimum_positive_fraction",
                        0.0,
                    ),
                }
            )
        PERINUCLEAR_COMBINATION = previous_config.get(
            "perinuclear_combination",
            previous_config.get("filters", {}).get(
                "perinuclear_combination",
                PERINUCLEAR_COMBINATION,
            ),
        )
    elif "map2_association" in previous_config:
        previous_map2 = previous_config["map2_association"]
        PERINUCLEAR_MARKERS = [
            {
                "name": previous_map2.get("channel", "MAP2"),
                "enabled": True,
                "ring_inner_um": previous_map2.get("ring_inner_um", 0.5),
                "ring_outer_um": previous_map2.get("ring_outer_um", 3.0),
                "pixel_threshold": previous_map2.get("pixel_threshold"),
                "initial_min_positive_fraction": previous_config.get(
                    "filters",
                    {},
                ).get("minimum_map2_ring_positive_fraction", 0.35),
            }
        ]
        PERINUCLEAR_COMBINATION = "any"

    validate_channel_configuration(CHANNELS, SEGMENTATION_CHANNEL)

    print(f"Previous result folder:\n{previous_result_folder}")
    print("Previous generalized settings restored.")

selected_file, _ = QFileDialog.getOpenFileName(
    None,
    "Select the original microscopy TIFF",
    "",
    "TIFF images (*.tif *.tiff);;All files (*.*)",
)
if not selected_file:
    raise RuntimeError("No input TIFF file was selected.")

# Do not resolve mapped Windows drives; resolving can produce a much longer UNC path.
DEFAULT_INPUT = Path(selected_file)
REPO_ROOT = DEFAULT_INPUT.parent

print(f"Selected input file:\n{DEFAULT_INPUT}")

## 4. Load and validate the microscopy image

The image is loaded once, standardized internally to `CZYX`, and then exposed as named `ZYX` channel volumes.

In [ ]:
manual_spacing = VOXEL_SPACING_OVERRIDE_UM
if RUN_MODE in {"resume", "density_only"} and previous_config is not None:
    manual_spacing = tuple(previous_config["calibration"]["voxel_spacing_zyx_um"])

VOXEL_SPACING_UM, CALIBRATION_SOURCE = read_voxel_spacing_um(
    DEFAULT_INPUT,
    manual_override=manual_spacing,
)
Z_SPACING_UM, Y_SPACING_UM, X_SPACING_UM = VOXEL_SPACING_UM
XY_SPACING_UM = float(np.mean((Y_SPACING_UM, X_SPACING_UM)))
ANISOTROPY = float(Z_SPACING_UM / XY_SPACING_UM)

image_czyx, image_information, OME_ZARR_CACHE_INFO = (
    load_tiff_with_optional_ome_zarr_cache(
        DEFAULT_INPUT,
        voxel_spacing_zyx_um=VOXEL_SPACING_UM,
        channel_configuration=CHANNELS,
        use_cache=USE_OME_ZARR_CACHE,
        cache_directory=OME_ZARR_CACHE_DIRECTORY,
        requested_chunks=OME_ZARR_CHUNKS_CZYX,
        axes_override=AXES_OVERRIDE,
        time_index=TIME_INDEX,
    )
)

channel_count = int(image_czyx.shape[0])
for item in CHANNELS:
    if int(item["index"]) >= channel_count:
        raise ValueError(
            f"Channel {item['name']!r} uses index {item['index']}, but the "
            f"standardized image has {channel_count} channels."
        )

channel_volumes = {
    item["name"]: np.ascontiguousarray(image_czyx[int(item["index"])])
    for item in CHANNELS
}
segmentation_volume = channel_volumes[SEGMENTATION_CHANNEL]
INPUT_SHA256 = file_sha256(DEFAULT_INPUT) if COMPUTE_INPUT_SHA256 else None

if RUN_MODE in {"resume", "density_only"} and previous_config is not None:
    expected_hash = previous_config["input"].get("sha256")
    if expected_hash and INPUT_SHA256 and expected_hash != INPUT_SHA256:
        raise ValueError(
            "The selected TIFF does not match the SHA-256 fingerprint stored "
            "with the previous analysis. Select the original TIFF."
        )

print("Original TIFF information:")
for key, value in image_information.items():
    print(f"  {key}: {value}")
print(f"Calibration source: {CALIBRATION_SOURCE}")
print(f"Voxel spacing (Z, Y, X): {VOXEL_SPACING_UM} µm")
print(f"Cellpose Z/XY anisotropy: {ANISOTROPY:.6f}")
print(f"Storage backend: {OME_ZARR_CACHE_INFO['storage_backend']}")

if OME_ZARR_CACHE_INFO["tiff_load_seconds"] is not None:
    print(f"TIFF loading: {OME_ZARR_CACHE_INFO['tiff_load_seconds']:.2f} s")
if OME_ZARR_CACHE_INFO["ome_zarr_conversion_seconds"] is not None:
    print(
        "OME-Zarr conversion: "
        f"{OME_ZARR_CACHE_INFO['ome_zarr_conversion_seconds']:.2f} s"
    )
if OME_ZARR_CACHE_INFO["ome_zarr_load_seconds"] is not None:
    print(
        "OME-Zarr loading: "
        f"{OME_ZARR_CACHE_INFO['ome_zarr_load_seconds']:.2f} s"
    )
if OME_ZARR_CACHE_INFO["enabled"]:
    print(f"OME-Zarr cache: {OME_ZARR_CACHE_INFO['path_at_analysis']}")
    print(f"OME-Zarr chunks CZYX: {OME_ZARR_CACHE_INFO['chunks_czyx']}")

print(f"Segmentation channel: {SEGMENTATION_CHANNEL}")
print(
    f"Segmentation intensity range: {segmentation_volume.min()} to "
    f"{segmentation_volume.max()} ({segmentation_volume.dtype})"
)


## 4b. MNTB ROI and calibrated sampling volume
The ROI is inferred from nonzero values in any original channel, filling enclosed XY holes plane by plane. This is an estimate of the Fiji crop, not automatic anatomical segmentation. Review all Z planes in Napari, edit the Labels layer if needed, and click **Accept reviewed MNTB ROI**. Exterior-connected dark areas and isolated exterior signal cannot be resolved reliably from intensity alone. An explicit binary Fiji mask is preferable.
Use `constant_xy` only if a single Fiji contour was applied to the whole stack. `per_slice` is the conservative default. Z indices are zero-based (Fiji slice 1 = Z 0). Active guards exclude surface planes even at zero guard, matching the existing whole-nucleus exclusion rule. Reported density is descriptive for retained nuclei; it is not an unbiased stereological estimate.


In [ ]:
# Reconstruct before segmentation, using all ORIGINAL image channels.
# No intensity normalization, dilation, or convex hull is applied.
roi_source = "inferred_" + MNTB_ROI_MODE
saved_roi = (previous_result_folder / "mntb_roi.tif"
             if previous_result_folder is not None else None)
if MNTB_ROI_MASK_PATH:
    roi_path = Path(MNTB_ROI_MASK_PATH)
    mntb_roi = np.asarray(tifffile.imread(roi_path)) > 0
    roi_source = "explicit_mask"
elif saved_roi is not None and saved_roi.is_file():
    mntb_roi = np.asarray(tifffile.imread(saved_roi)) > 0
    roi_source = "restored_mask"
else:
    mntb_roi = reconstruct_mntb_roi(image_czyx, MNTB_ROI_MODE)
if mntb_roi.shape != segmentation_volume.shape:
    raise ValueError("MNTB ROI must match the original image ZYX grid exactly.")
if not mntb_roi.any():
    raise ValueError("Empty MNTB ROI. Check the input or supply an explicit ROI mask.")
print(f"MNTB ROI source: {roi_source}")
print(f"Provisional ROI voxels: {int(mntb_roi.sum()):,}")
print(f"Provisional ROI volume: {mntb_roi.sum() * np.prod(VOXEL_SPACING_UM):,.2f} µm³")
print("Review MNTB ROI in Napari across Z; dark regions connected to the exterior cannot be recovered reliably.")


In [ ]:
# Lightweight retrospective workflow: no model, morphology, or shell measurements.
if RUN_MODE == "density_only":
    saved_properties_path = previous_result_folder / "retained_mask_properties.csv"
    if not saved_properties_path.is_file():
        raise FileNotFoundError("density_only requires retained_mask_properties.csv from a previous export.")
    saved_retained = pd.read_csv(saved_properties_path).set_index("label")
    saved_settings = dict(previous_config["filters"])
    for column in ("centroid_z_px", "centroid_y_px", "centroid_x_px"):
        if column not in saved_retained:
            raise ValueError(f"Previous table lacks {column}; density_only cannot determine ROI membership.")
    saved_filtered_path = previous_result_folder / "filtered_masks_syglass.tif"
    saved_display_masks = np.asarray(tifffile.imread(
        saved_filtered_path if saved_filtered_path.is_file() else previous_masks_path))
    if saved_display_masks.shape != segmentation_volume.shape:
        raise ValueError("Previous masks do not match the selected image ZYX grid.")
    if not saved_filtered_path.is_file():
        saved_display_masks = np.where(np.isin(saved_display_masks, saved_retained.index), saved_display_masks, 0)
    if "viewer" in globals():
        try:
            viewer.close()
        except Exception:
            pass
    viewer = napari.Viewer()
    for item in CHANNELS:
        viewer.add_image(channel_volumes[item["name"]], name=item["name"],
                         scale=VOXEL_SPACING_UM, visible=item["name"] == SEGMENTATION_CHANNEL)
    viewer.add_labels(saved_display_masks, name="previously retained nuclei", scale=VOXEL_SPACING_UM)
    retrospective_roi_layer = viewer.add_labels(mntb_roi.astype(np.uint8), name="MNTB ROI - review",
                                                scale=VOXEL_SPACING_UM, opacity=0.25)
    retrospective_info = Label(value=f"Previous retained nuclei: {len(saved_retained)}. Review ROI across Z.")
    retrospective_status = Label(value="Density: pending ROI review")
    retrospective_accept = PushButton(text="Accept reviewed ROI and calculate density")
    retrospective_save = PushButton(text="Save density addendum")
    retrospective_reviewed = False
    retrospective_result = None

    def invalidate_retrospective_roi(*_):
        global retrospective_reviewed
        retrospective_reviewed = False
        retrospective_status.value = "ROI changed: accept again before saving"

    def calculate_retrospective_density(*_):
        global retrospective_reviewed, retrospective_result, retrospective_counts, retrospective_table
        roi = np.asarray(retrospective_roi_layer.data) > 0
        if roi.shape != segmentation_volume.shape or not roi.any():
            retrospective_reviewed = False
            retrospective_status.value = "Invalid or empty ROI"
            return
        inside, centroid_z = roi_centroid_membership(saved_retained, roi)
        _, sampling = density_z_planes(saved_settings, roi.shape[0], VOXEL_SPACING_UM[0])
        valid_z = (centroid_z >= 0) & (centroid_z < roi.shape[0])
        in_z = np.zeros(len(saved_retained), dtype=bool)
        in_z[valid_z] = sampling[centroid_z[valid_z]]
        counted = inside & in_z
        retrospective_counts = np.count_nonzero(roi, axis=(1, 2))
        retrospective_result = calculate_density_summary(retrospective_counts, saved_settings,
                                                         VOXEL_SPACING_UM, int(counted.sum()), True)
        retrospective_result.update({"previous_retained_nuclei": len(saved_retained),
                                     "previous_retained_outside_sampling_roi": int((~counted).sum()),
                                     "roi_source": roi_source,
                                     "source_analysis": str(previous_result_folder),
                                     "input_file": str(DEFAULT_INPUT)})
        retrospective_table = saved_retained.copy()
        retrospective_table["included_in_density"] = counted
        retrospective_reviewed = True
        density = retrospective_result["retained_nuclei_per_mm3"]
        value = (f"{density / 1000:,.1f} nuclei per (100 µm)³" if density is not None else "unavailable (empty volume)")
        retrospective_status.value = (f"Previously retained nuclei: {len(saved_retained):,}\n"+ format_density_panel(retrospective_result))

    def save_retrospective_density(*_):
        if not retrospective_reviewed:
            retrospective_status.value = "Accept the reviewed ROI before saving"
            return
        calculate_retrospective_density()
        if not retrospective_reviewed:
            return
        destination = previous_result_folder / ("density_addendum_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
        destination.mkdir(parents=True, exist_ok=False)
        roi = np.asarray(retrospective_roi_layer.data) > 0
        calibrated_label_tiff(destination / "mntb_roi.tif", roi.astype(np.uint8), VOXEL_SPACING_UM)
        pd.DataFrame([retrospective_result]).to_csv(destination / "mntb_density_summary.csv", index=False)
        retrospective_table.reset_index().to_csv(destination / "previous_nuclei_density_membership.csv", index=False)
        _, sampling = density_z_planes(saved_settings, roi.shape[0], VOXEL_SPACING_UM[0])
        pd.DataFrame({"z_index": np.arange(roi.shape[0]), "roi_voxels": retrospective_counts,
                      "area_um2": retrospective_counts * VOXEL_SPACING_UM[1] * VOXEL_SPACING_UM[2],
                      "included_in_density": sampling}).to_csv(destination / "mntb_roi_by_z.csv", index=False)
        record = {"density": retrospective_result, "original_filters": saved_settings,
                  "voxel_spacing_zyx_um": list(VOXEL_SPACING_UM), "roi_mode": MNTB_ROI_MODE,
                  "input_sha256": INPUT_SHA256, "source_configuration": previous_config,
                  "date": datetime.now().isoformat()}
        (destination / "density_addendum.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
        retrospective_status.value = f"Saved: {destination.name}"
        print(f"Density addendum saved to: {destination}")

    retrospective_roi_layer.events.data.connect(invalidate_retrospective_roi)
    if hasattr(retrospective_roi_layer.events, "paint"):
        retrospective_roi_layer.events.paint.connect(invalidate_retrospective_roi)
    retrospective_accept.changed.connect(calculate_retrospective_density)
    retrospective_save.changed.connect(save_retrospective_density)
    retrospective_panel = Container(widgets=[retrospective_info, retrospective_status,
                                              retrospective_accept, retrospective_save])
    viewer.window.add_dock_widget(retrospective_panel, area="right", name="Previous analysis: density only")
    print("Review ROI, accept, then Save density addendum. Later analysis cells are skipped in density_only mode.")


## 5. Run Cellpose-SAM or load raw masks

`CELLPOSE_MIN_SIZE_VOXELS` remains a Cellpose voxel-based parameter. The later live filter uses calibrated physical volume in µm³.

In [ ]:
if RUN_MODE != "density_only":
    # --------------------------------------------------------
    # Select the processing device
    # --------------------------------------------------------

    if torch.cuda.is_available():

        DEVICE = torch.device("cuda")

        print("Acceleration backend: NVIDIA CUDA")
        print("GPU:", torch.cuda.get_device_name(0))


    elif (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_built()
        and torch.backends.mps.is_available()
    ):

        DEVICE = torch.device("mps")

        print("Acceleration backend: Apple MPS")
        print("GPU: Apple Silicon integrated GPU")


    else:

        DEVICE = torch.device("cpu")

        print("Acceleration backend: CPU")
        print("No compatible GPU acceleration was detected.")


    USING_ACCELERATOR = DEVICE.type in {
        "cuda",
        "mps",
    }

    print("Selected PyTorch device:", DEVICE)


    def synchronize_selected_device():

        if DEVICE.type == "cuda":

            torch.cuda.synchronize()

        elif (
            DEVICE.type == "mps"
            and hasattr(torch, "mps")
            and hasattr(torch.mps, "synchronize")
        ):

            torch.mps.synchronize()


    raw_3d_probs = None
    CELLPOSE_MODEL_LOAD_SECONDS = None
    CELLPOSE_EVAL_SECONDS = None


    # --------------------------------------------------------
    # Resume previous segmentation
    # --------------------------------------------------------

    if RUN_MODE in {"resume", "density_only"}:

        resume_start = perf_counter()

        masks = tifffile.imread(
            previous_result_folder
            / "cellpose_raw_masks.tif"
        )

        if masks.ndim != 3:

            raise ValueError(
                f"Expected a ZYX raw mask; "
                f"received {masks.shape}."
            )

        if masks.shape != segmentation_volume.shape:

            raise ValueError(
                f"Raw mask shape {masks.shape} "
                f"does not match image shape "
                f"{segmentation_volume.shape}."
            )

        print(
            "Loaded previous raw Cellpose masks "
            "without rerunning the model in "
            f"{perf_counter() - resume_start:.2f} s."
        )


    # --------------------------------------------------------
    # Run a new segmentation
    # --------------------------------------------------------

    else:

        # Cellpose file logging is optional.
        if not globals().get(
            "_CELLPOSE_LOGGER_INITIALIZED",
            False,
        ):

            try:

                io.logger_setup()

                print(
                    "Cellpose logging initialized.",
                    flush=True,
                )

            except PermissionError:

                print(
                    "Cellpose run.log is currently in use. "
                    "Continuing without resetting it.",
                    flush=True,
                )

            _CELLPOSE_LOGGER_INITIALIZED = True


        # Clear unused Apple GPU memory before loading the model.
        if DEVICE.type == "mps":

            torch.mps.empty_cache()


        # ----------------------------------------------------
        # Load the model
        # ----------------------------------------------------

        print(
            f"Loading Cellpose model: "
            f"{MODEL_NAME}...",
            flush=True,
        )

        model_start = perf_counter()

        model = models.CellposeModel(
            device=DEVICE,
            pretrained_model=MODEL_NAME,
        )

        synchronize_selected_device()

        CELLPOSE_MODEL_LOAD_SECONDS = float(
            perf_counter() - model_start
        )

        print(
            "Cellpose model loaded successfully "
            f"in {CELLPOSE_MODEL_LOAD_SECONDS:.2f} s.",
            flush=True,
        )


        # ----------------------------------------------------
        # Cellpose parameters
        # ----------------------------------------------------

        eval_parameters = {
            "z_axis": 0,
            "channel_axis": None,
            "do_3D": True,
            "anisotropy": ANISOTROPY,
            "batch_size": BATCH_SIZE,
            "cellprob_threshold": CELLPROB_THRESHOLD,
            "min_size": CELLPOSE_MIN_SIZE_VOXELS,
            "progress": True,
        }


        if FLOW3D_SMOOTH is not None:

            eval_parameters[
                "flow3D_smooth"
            ] = FLOW3D_SMOOTH


        if DIAMETER_PIXELS is not None:

            eval_parameters[
                "diameter"
            ] = DIAMETER_PIXELS


        # ----------------------------------------------------
        # Run 3D segmentation
        # ----------------------------------------------------

        print(
            "Starting full-resolution 3D "
            f"segmentation of "
            f"{segmentation_volume.shape}...",
            flush=True,
        )

        print(
            f"Device: {DEVICE}",
            flush=True,
        )

        print(
            f"Batch size: {BATCH_SIZE}",
            flush=True,
        )

        print(
            f"Anisotropy: {ANISOTROPY:.4f}",
            flush=True,
        )

        synchronize_selected_device()

        evaluation_start = perf_counter()

        masks, flows, styles = model.eval(
            segmentation_volume,
            **eval_parameters,
        )

        synchronize_selected_device()

        CELLPOSE_EVAL_SECONDS = float(
            perf_counter() - evaluation_start
        )

        print(
            "Cellpose evaluation completed in "
            f"{CELLPOSE_EVAL_SECONDS / 60:.2f} min.",
            flush=True,
        )


        # ----------------------------------------------------
        # Recover the probability volume when available
        # ----------------------------------------------------

        try:

            candidate_probs = np.asarray(
                flows[2]
            )

            candidate_probs = np.squeeze(
                candidate_probs
            )

            if candidate_probs.shape == masks.shape:

                raw_3d_probs = candidate_probs

            else:

                print(
                    "The Cellpose probability output "
                    f"has shape {candidate_probs.shape}, "
                    f"but the masks have shape "
                    f"{masks.shape}. The probability "
                    "layer will not be added."
                )

        except (
            IndexError,
            TypeError,
            ValueError,
        ):

            raw_3d_probs = None

            print(
                "The Cellpose probability volume "
                "could not be recovered. The masks "
                "remain available."
            )


    # --------------------------------------------------------
    # Validate the segmentation
    # --------------------------------------------------------

    masks = np.asarray(
        masks
    )


    if masks.shape != segmentation_volume.shape:

        raise ValueError(
            f"Mask shape {masks.shape} "
            f"does not match segmentation volume "
            f"{segmentation_volume.shape}."
        )


    if int(masks.max()) == 0:

        raise RuntimeError(
            "Cellpose produced zero masks."
        )


    detected_labels = np.unique(
        masks
    )

    detected_labels = detected_labels[
        detected_labels != 0
    ]


    print(
        "Mask shape:",
        masks.shape,
    )

    print(
        "Maximum label ID:",
        int(masks.max()),
    )

    print(
        "Detected labels:",
        len(detected_labels),
    )

## 6. Inspect the raw segmentation in Napari

The segmentation channel is initially visible. Other staining channels can be toggled in Napari. Raw masks are not modified by later filters.

In [ ]:
if RUN_MODE != "density_only":
    if "viewer" in globals():
        try:
            viewer.close()
        except Exception:
            pass

    viewer = napari.Viewer()

    for item in CHANNELS:
        viewer.add_image(
            channel_volumes[item["name"]],
            name=f"channel {item['index']}: {item['name']}",
            scale=VOXEL_SPACING_UM,
            colormap=item.get("colormap", "gray"),
            visible=(item["name"] == SEGMENTATION_CHANNEL),
        )

    if raw_3d_probs is not None:
        viewer.add_image(
            raw_3d_probs,
            name="Cellpose probability",
            scale=VOXEL_SPACING_UM,
            visible=False,
        )

    viewer.add_labels(
        masks,
        name="raw Cellpose masks",
        scale=VOXEL_SPACING_UM,
        opacity=0.70,
    )
    mntb_roi_layer = viewer.add_labels(
        mntb_roi.astype(np.uint8), name="MNTB ROI - review before density",
        scale=VOXEL_SPACING_UM, opacity=0.25, visible=True,
    )


## 7. Calculate morphology and staining measurements

Raw, background-corrected, and distribution-based intensity statistics are calculated for every channel with `measure=True`. Filtering uses either raw mean or background-corrected mean according to `INTENSITY_MODE`.

In [ ]:
if RUN_MODE != "density_only":
    mask_properties, background_table = calculate_mask_properties(
        masks=masks,
        channel_volumes=channel_volumes,
        channel_configuration=CHANNELS,
        voxel_spacing=VOXEL_SPACING_UM,
        background_percentile=GLOBAL_BACKGROUND_PERCENTILE,
    )

    if INTENSITY_MODE == "raw":
        intensity_column_prefix = "mean_intensity_"
    elif INTENSITY_MODE == "global_background_subtracted":
        intensity_column_prefix = "background_corrected_mean_"
    else:
        raise ValueError(
            'INTENSITY_MODE must be "raw" or '
            '"global_background_subtracted".'
        )

    print(f"Measured {len(mask_properties)} masks.")
    display(mask_properties.describe().T)
    display(background_table)

## 8. Measure configurable perinuclear markers

Cellpose has segmented DAPI alone. This section measures each enabled marker independently in its own physical 3D shell outside every nucleus. Direct DAPI–marker pixel overlap is not required because cytoplasmic and synaptic markers may be absent from the nuclear interior.

The marker-specific mean, median, 75th percentile, positive voxel fraction, resolved threshold, and shell geometry are retained. Multiple marker rules are combined later with `any` or `all` in the live filter.


In [ ]:
if RUN_MODE != "density_only":
    if SEGMENTATION_CHANNEL != "DAPI":
        raise ValueError(
            "This perinuclear-marker workflow requires DAPI as the segmentation channel."
        )
    if PERINUCLEAR_COMBINATION not in {"any", "all"}:
        raise ValueError('PERINUCLEAR_COMBINATION must be "any" or "all".')

    enabled_perinuclear_markers = [
        dict(item) for item in PERINUCLEAR_MARKERS if item.get("enabled", True)
    ]
    if not enabled_perinuclear_markers:
        raise ValueError("Enable at least one item in PERINUCLEAR_MARKERS.")

    marker_names = [str(item["name"]) for item in enabled_perinuclear_markers]
    marker_keys = [safe_key(name).lower() for name in marker_names]
    if len(marker_names) != len(set(marker_names)):
        raise ValueError("Enabled perinuclear marker names must be unique.")
    if len(marker_keys) != len(set(marker_keys)):
        raise ValueError(
            "Enabled marker names must remain unique after CSV-safe conversion."
        )

    PERINUCLEAR_MARKER_RESULTS = {}
    perinuclear_metric_tables = []

    for marker_configuration in enabled_perinuclear_markers:
        marker_name = str(marker_configuration["name"])
        if marker_name not in channel_volumes:
            raise KeyError(
                f"Perinuclear marker {marker_name!r} is not present in CHANNELS."
            )

        ring_inner_um = float(marker_configuration.get("ring_inner_um", 0.5))
        ring_outer_um = float(marker_configuration.get("ring_outer_um", 3.0))
        minimum_fraction = float(
            marker_configuration.get("initial_min_positive_fraction", 0.0)
        )
        if not 0 <= minimum_fraction <= 1:
            raise ValueError(
                f"{marker_name} initial_min_positive_fraction must be between 0 and 1."
            )

        metrics, ring_labels, resolved_threshold, threshold_source = (
            calculate_perinuclear_marker_metrics(
                nucleus_masks=masks,
                intensity_volume=channel_volumes[marker_name],
                voxel_spacing=VOXEL_SPACING_UM,
                channel_name=marker_name,
                inner_distance_um=ring_inner_um,
                outer_distance_um=ring_outer_um,
                pixel_threshold=marker_configuration.get("pixel_threshold"),
            )
        )
        prefix = safe_key(marker_name).lower()
        fraction_column = f"{prefix}_ring_positive_fraction"
        perinuclear_metric_tables.append(metrics)
        mask_properties = mask_properties.join(
            metrics,
            how="left",
            validate="one_to_one",
        )
        mask_properties[fraction_column] = mask_properties[fraction_column].fillna(0.0)

        initial_keep = mask_properties[fraction_column] >= minimum_fraction
        initial_labels = mask_properties.index[initial_keep].to_numpy()
        initial_associated_masks = np.where(
            np.isin(masks, initial_labels),
            masks,
            0,
        )

        shell_layer_name = f"{marker_name} perinuclear shells"
        associated_layer_name = f"initial {marker_name}-associated DAPI nuclei"
        for layer_name in [shell_layer_name, associated_layer_name]:
            if layer_name in viewer.layers:
                viewer.layers.remove(layer_name)

        viewer.add_labels(
            ring_labels,
            name=shell_layer_name,
            scale=VOXEL_SPACING_UM,
            opacity=0.45,
            visible=False,
        )
        viewer.add_labels(
            initial_associated_masks,
            name=associated_layer_name,
            scale=VOXEL_SPACING_UM,
            opacity=0.75,
            visible=(len(PERINUCLEAR_MARKER_RESULTS) == 0),
        )

        PERINUCLEAR_MARKER_RESULTS[marker_name] = {
            "name": marker_name,
            "prefix": prefix,
            "fraction_column": fraction_column,
            "ring_inner_um": ring_inner_um,
            "ring_outer_um": ring_outer_um,
            "configured_pixel_threshold": marker_configuration.get("pixel_threshold"),
            "resolved_pixel_threshold": float(resolved_threshold),
            "threshold_source": threshold_source,
            "initial_min_positive_fraction": minimum_fraction,
            "ring_labels": ring_labels,
        }

        print(
            f"{marker_name}: threshold {resolved_threshold:.4f} "
            f"({threshold_source}); shell {ring_inner_um:.2f}–{ring_outer_um:.2f} µm; "
            f"initially associated {int(initial_keep.sum())}/{len(initial_keep)} nuclei."
        )

    perinuclear_metrics = pd.concat(perinuclear_metric_tables, axis=1)
    display(perinuclear_metrics.describe().T)


In [ ]:
if RUN_MODE != "density_only":
    #------ DAPI INTENSITY THROUGH Z

    dapi_p50_by_z = np.percentile(
        segmentation_volume,
        50,
        axis=(1, 2),
    )

    dapi_p99_by_z = np.percentile(
        segmentation_volume,
        99,
        axis=(1, 2),
    )

    plt.figure(figsize=(8, 4))
    plt.plot(dapi_p50_by_z, label="DAPI median")
    plt.plot(dapi_p99_by_z, label="DAPI 99th percentile")
    plt.xlabel("Z plane")
    plt.ylabel("Raw intensity")
    plt.title("DAPI intensity through the Z-stack")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

## 9. Initial quality control

These plots help identify debris, very large fused masks, and unusual morphology before setting thresholds.

In [ ]:
if RUN_MODE != "density_only":
    figure, axes = plt.subplots(1, 3, figsize=(17, 4.5))

    axes[0].hist(
        mask_properties["volume_um3"].dropna(),
        bins=40,
        color="steelblue",
        edgecolor="black",
        alpha=0.80,
    )
    axes[0].set_xlabel("Mask volume (µm³)")
    axes[0].set_ylabel("Number of masks")
    axes[0].set_title("All raw Cellpose masks")
    axes[0].grid(alpha=0.2)

    axes[1].scatter(
        mask_properties["volume_um3"],
        mask_properties["sphericity"],
        s=14,
        alpha=0.45,
        color="slateblue",
    )
    axes[1].set_xlabel("Mask volume (µm³)")
    axes[1].set_ylabel("Sphericity")
    axes[1].set_title("Volume versus sphericity")
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(alpha=0.2)

    for marker_name, marker_info in PERINUCLEAR_MARKER_RESULTS.items():
        column = marker_info["fraction_column"]
        minimum = marker_info["initial_min_positive_fraction"]
        axes[2].hist(
            mask_properties[column].dropna(),
            bins=np.linspace(0, 1, 41),
            alpha=0.45,
            label=marker_name,
        )
        axes[2].axvline(minimum, linestyle="--", linewidth=1.5)

    axes[2].set_xlabel("Marker-positive perinuclear-shell fraction")
    axes[2].set_ylabel("Number of nuclei")
    axes[2].set_title("Perinuclear marker association")
    axes[2].set_xlim(0, 1)
    axes[2].legend()
    axes[2].grid(alpha=0.2)

    figure.tight_layout()
    plt.show()


## 10. Dynamic live filtering panel

The panel is created from `CHANNELS`, so it can contain zero, one, two, or many marker filters without rewriting this cell.

Morphology and Z-guard criteria are always combined with `AND`. The `all/any` selector controls only how multiple intensity-channel criteria are combined.

In [ ]:
if RUN_MODE != "density_only":
    filter_properties = mask_properties.copy()
    max_z_index = int(masks.shape[0] - 1)
    z_spacing_um = float(VOXEL_SPACING_UM[0])
    voxel_volume_um3 = float(np.prod(VOXEL_SPACING_UM))

    default_last_z = (
        max_z_index
        if INITIAL_LAST_TISSUE_Z is None
        else int(INITIAL_LAST_TISSUE_Z)
    )
    default_min_volume = (
        float(500 * voxel_volume_um3)
        if INITIAL_MIN_VOLUME_UM3 is None
        else float(INITIAL_MIN_VOLUME_UM3)
    )

    first_z_widget = Slider(
        value=int(np.clip(INITIAL_FIRST_TISSUE_Z, 0, max_z_index)),
        min=0,
        max=max_z_index,
        step=1,
        label="first tissue Z",
    )
    last_z_widget = Slider(
        value=int(np.clip(default_last_z, 0, max_z_index)),
        min=0,
        max=max_z_index,
        step=1,
        label="last tissue Z",
    )
    max_guard_um = max(1.0, float(np.ceil(max_z_index * z_spacing_um / 2)))
    guard_widget = FloatSlider(
        value=float(np.clip(INITIAL_Z_GUARD_UM, 0, max_guard_um)),
        min=0.0,
        max=max_guard_um,
        step=0.1,
        label="Z guard (µm)",
    )
    guard_mode_widget = ComboBox(
        value=INITIAL_Z_GUARD_MODE,
        choices=["none", "first", "last", "both"],
        label="Z guard mode",
    )

    maximum_volume = float(np.ceil(filter_properties["volume_um3"].max()))
    volume_widget = FloatSlider(
        value=float(np.clip(default_min_volume, 0, maximum_volume)),
        min=0.0,
        max=maximum_volume,
        step=max(0.1, maximum_volume / 1000),
        label="minimum volume (µm³)",
    )
    sphericity_widget = FloatSlider(
        value=float(INITIAL_MIN_SPHERICITY),
        min=0.0,
        max=1.0,
        step=0.01,
        label="minimum sphericity",
    )

    perinuclear_combination_widget = ComboBox(
        value=PERINUCLEAR_COMBINATION,
        choices=["any", "all"],
        label="combine perinuclear markers",
    )
    perinuclear_widgets = {}
    perinuclear_widget_list = []
    for marker_name, marker_info in PERINUCLEAR_MARKER_RESULTS.items():
        widget = FloatSlider(
            value=float(marker_info["initial_min_positive_fraction"]),
            min=0.0,
            max=1.0,
            step=0.01,
            label=f"minimum {marker_name}-positive ring fraction",
        )
        perinuclear_widgets[marker_name] = {
            "column": marker_info["fraction_column"],
            "minimum": widget,
        }
        perinuclear_widget_list.append(widget)

    combination_widget = ComboBox(
        value=FILTER_COMBINATION,
        choices=["all", "any"],
        label="combine nuclear intensity filters",
    )
    intensity_widgets = {}
    intensity_widget_list = []
    for item in CHANNELS:
        if not item.get("filter_enabled", False):
            continue
        name = item["name"]
        key = safe_key(name)
        column = f"{intensity_column_prefix}{key}"
        if column not in filter_properties:
            raise KeyError(
                f"Filtering was enabled for {name!r}, but {column!r} was not measured."
            )
        observed_min = float(np.floor(filter_properties[column].min()))
        observed_max = float(np.ceil(filter_properties[column].max()))
        if observed_max <= observed_min:
            observed_max = observed_min + 1.0
        step = max(0.01, (observed_max - observed_min) / 500)
        configured_min = item.get("filter_min")
        configured_max = item.get("filter_max")
        start_min = observed_min if configured_min is None else float(configured_min)
        start_max = observed_max if configured_max is None else float(configured_max)
        minimum_widget = FloatSlider(
            value=float(np.clip(start_min, observed_min, observed_max)),
            min=observed_min,
            max=observed_max,
            step=step,
            label=f"{name} minimum",
        )
        maximum_widget = FloatSlider(
            value=float(np.clip(start_max, observed_min, observed_max)),
            min=observed_min,
            max=observed_max,
            step=step,
            label=f"{name} maximum",
        )
        intensity_widgets[name] = {
            "column": column,
            "minimum": minimum_widget,
            "maximum": maximum_widget,
        }
        intensity_widget_list.extend([minimum_widget, maximum_widget])

    count_label = Label(value="Masks kept: --")
    guard_count_label = Label(value="Z guard excluded: --")
    status_label = Label(value="Ready")
    refresh_button = PushButton(text="Refresh filters")

    filter_panel = Container(
        widgets=[
            first_z_widget,
            last_z_widget,
            guard_widget,
            guard_mode_widget,
            volume_widget,
            sphericity_widget,
            perinuclear_combination_widget,
            *perinuclear_widget_list,
            combination_widget,
            *intensity_widget_list,
            count_label,
            guard_count_label,
            status_label,
            refresh_button,
        ],
        layout="vertical",
    )

    live_layer_name = "live filtered masks"
    guard_layer_name = "excluded by Z guard"
    current_keep = None
    filtered_masks = None


    def current_filter_settings():
        return {
            "minimum_volume_um3": float(volume_widget.value),
            "minimum_sphericity": float(sphericity_widget.value),
            "first_tissue_z": int(first_z_widget.value),
            "last_tissue_z": int(last_z_widget.value),
            "z_guard_um": float(guard_widget.value),
            "z_guard_mode": str(guard_mode_widget.value),
            "perinuclear_combination": str(perinuclear_combination_widget.value),
            "perinuclear_filters": {
                name: {
                    "column": widgets["column"],
                    "minimum_positive_fraction": float(widgets["minimum"].value),
                }
                for name, widgets in perinuclear_widgets.items()
            },
            "intensity_combination": str(combination_widget.value),
            "intensity_filters": {
                name: {
                    "column": widgets["column"],
                    "minimum": float(widgets["minimum"].value),
                    "maximum": float(widgets["maximum"].value),
                }
                for name, widgets in intensity_widgets.items()
            },
        }


    def calculate_perinuclear_filter_keep(settings, properties=None):
        properties = filter_properties if properties is None else properties
        conditions = []
        for marker_filter in settings["perinuclear_filters"].values():
            conditions.append(
                properties[marker_filter["column"]].fillna(0.0)
                >= marker_filter["minimum_positive_fraction"]
            )
        if not conditions:
            return pd.Series(True, index=properties.index)
        condition_table = pd.concat(conditions, axis=1)
        if settings["perinuclear_combination"] == "all":
            return condition_table.all(axis=1)
        if settings["perinuclear_combination"] == "any":
            return condition_table.any(axis=1)
        raise ValueError('Perinuclear combination must be "any" or "all".')


    def apply_filter_settings(settings):
        z_properties = calculate_z_guard_properties(
            filter_properties,
            first_tissue_z=settings["first_tissue_z"],
            last_tissue_z=settings["last_tissue_z"],
            guard_um=settings["z_guard_um"],
            guard_mode=settings["z_guard_mode"],
            z_spacing_um=z_spacing_um,
            max_z_index=max_z_index,
        )
        for column in z_properties.columns:
            filter_properties[column] = z_properties[column]
            mask_properties[column] = z_properties[column]

        morphology_keep = (
            (filter_properties["volume_um3"] >= settings["minimum_volume_um3"])
            & (filter_properties["sphericity"] >= settings["minimum_sphericity"])
            & (~filter_properties["excluded_by_z_guard"])
        )
        perinuclear_keep = calculate_perinuclear_filter_keep(settings)

        intensity_conditions = []
        for name, intensity_filter in settings["intensity_filters"].items():
            if intensity_filter["minimum"] > intensity_filter["maximum"]:
                raise ValueError(f"{name} minimum intensity exceeds its maximum.")
            intensity_conditions.append(
                filter_properties[intensity_filter["column"]].between(
                    intensity_filter["minimum"],
                    intensity_filter["maximum"],
                    inclusive="both",
                )
            )
        if not intensity_conditions:
            intensity_keep = pd.Series(True, index=filter_properties.index)
        elif settings["intensity_combination"] == "all":
            intensity_keep = pd.concat(intensity_conditions, axis=1).all(axis=1)
        else:
            intensity_keep = pd.concat(intensity_conditions, axis=1).any(axis=1)

        _, sampling_z = density_z_planes(settings, masks.shape[0], z_spacing_um)
        z_valid = (roi_centroid_z >= 0) & (roi_centroid_z < masks.shape[0])
        centroid_in_z = np.zeros(len(filter_properties), dtype=bool)
        centroid_in_z[z_valid] = sampling_z[roi_centroid_z[z_valid]]
        filter_properties["centroid_in_mntb_roi"] = roi_inside
        filter_properties["centroid_in_sampling_z"] = centroid_in_z
        filter_properties["excluded_by_sampling_roi"] = ~(roi_inside & centroid_in_z)
        return morphology_keep & perinuclear_keep & intensity_keep & roi_inside & centroid_in_z


    def refresh_live_filter(*_):
        global current_keep, filtered_masks
        try:
            settings = current_filter_settings()
            current_keep = apply_filter_settings(settings)
            kept_labels = filter_properties.index[current_keep].to_numpy()
            filtered_masks = np.where(np.isin(masks, kept_labels), masks, 0)

            # Every shell voxel carries its parent nucleus label. Reuse the same
            # kept-label lookup so the live shell layers always match the live
            # filtered nuclei without recalculating shell geometry or intensities.
            kept_label_lookup = np.zeros(int(masks.max()) + 1, dtype=bool)
            kept_label_lookup[kept_labels.astype(int)] = True
            first_marker_name = next(iter(PERINUCLEAR_MARKER_RESULTS), None)
            for marker_name, marker_info in PERINUCLEAR_MARKER_RESULTS.items():
                ring_labels = marker_info["ring_labels"]
                live_ring_labels = np.where(
                    kept_label_lookup[ring_labels],
                    ring_labels,
                    0,
                )
                live_ring_layer_name = f"live {marker_name} perinuclear shells"
                if live_ring_layer_name in viewer.layers:
                    viewer.layers[live_ring_layer_name].data = live_ring_labels
                    viewer.layers[live_ring_layer_name].scale = VOXEL_SPACING_UM
                else:
                    viewer.add_labels(
                        live_ring_labels,
                        name=live_ring_layer_name,
                        scale=VOXEL_SPACING_UM,
                        opacity=0.55,
                        visible=(marker_name == first_marker_name),
                    )

            guard_excluded = filter_properties["excluded_by_z_guard"].astype(bool)
            guard_labels = filter_properties.index[guard_excluded].to_numpy()
            guard_masks = np.where(np.isin(masks, guard_labels), masks, 0)

            if live_layer_name in viewer.layers:
                viewer.layers[live_layer_name].data = filtered_masks
                viewer.layers[live_layer_name].scale = VOXEL_SPACING_UM
            else:
                viewer.add_labels(
                    filtered_masks,
                    name=live_layer_name,
                    scale=VOXEL_SPACING_UM,
                    opacity=0.75,
                )
            if guard_layer_name in viewer.layers:
                viewer.layers[guard_layer_name].data = guard_masks
                viewer.layers[guard_layer_name].scale = VOXEL_SPACING_UM
            else:
                viewer.add_labels(
                    guard_masks,
                    name=guard_layer_name,
                    scale=VOXEL_SPACING_UM,
                    opacity=0.75,
                    visible=False,
                )

            count_label.value = (
                f"Masks kept: {int(current_keep.sum())} / {len(current_keep)}"
            )
            guard_count_label.value = (
                f"Z guard excluded: {int(guard_excluded.sum())}"
            )
            density_result = calculate_density_summary(roi_plane_counts, settings, VOXEL_SPACING_UM, int(current_keep.sum()), roi_reviewed)
            roi_volume_label.value = format_density_panel(density_result)
            density_label.value = ""
            density = density_result["retained_nuclei_per_mm3"]
            #density_label.value = (f"Nuclei per (100 µm)³: {density / 1000:,.1f}"
                                    #if density is not None
                                    #else f"Density unavailable: {density_result['density_status']}")
            status_label.value = "Valid filter settings"
            viewer.status = (
                f"Kept {int(current_keep.sum())}/{len(current_keep)} | "
                f"perinuclear markers: {settings['perinuclear_combination']} | "
                f"volume ≥ {settings['minimum_volume_um3']:.1f} µm³ | "
                f"sphericity ≥ {settings['minimum_sphericity']:.2f}"
            )
        except ValueError as error:
            density_label.value = "Density unavailable: invalid settings"
            status_label.value = f"Invalid settings: {error}"
            viewer.status = str(error)


    for widget in [
        first_z_widget,
        last_z_widget,
        guard_widget,
        guard_mode_widget,
        volume_widget,
        sphericity_widget,
        perinuclear_combination_widget,
        *perinuclear_widget_list,
        combination_widget,
        *intensity_widget_list,
    ]:
        widget.changed.connect(refresh_live_filter)
    refresh_button.changed.connect(refresh_live_filter)

    if "filter_dock_widget" in globals():
        try:
            viewer.window.remove_dock_widget(filter_dock_widget)
        except (KeyError, RuntimeError):
            pass

    filter_dock_widget = viewer.window.add_dock_widget(
        filter_panel,
        area="right",
        name="DAPI nuclei and perinuclear markers",
    )

    # ROI counts are cached; slider changes only sum one small array per Z.
    mntb_roi_layer = viewer.layers["MNTB ROI - review before density"]
    roi_reviewed = False
    roi_plane_counts = np.count_nonzero(mntb_roi_layer.data, axis=(1, 2))
    roi_inside, roi_centroid_z = roi_centroid_membership(filter_properties, mntb_roi_layer.data > 0)
    roi_volume_label = Label(value="ROI volume: --")
    density_label = Label(value="Density: review ROI first")
    roi_review_button = PushButton(text="Accept reviewed MNTB ROI")
    filter_panel.append(roi_volume_label)
    filter_panel.append(density_label)
    filter_panel.append(roi_review_button)


    def accept_mntb_roi(*_):
        global mntb_roi, roi_reviewed, roi_plane_counts, roi_inside, roi_centroid_z
        mntb_roi = np.asarray(mntb_roi_layer.data) > 0
        if mntb_roi.shape != masks.shape or not mntb_roi.any():
            roi_reviewed = False
            density_label.value = "Invalid or empty ROI"
            return
        roi_plane_counts = np.count_nonzero(mntb_roi, axis=(1, 2))
        roi_inside, roi_centroid_z = roi_centroid_membership(filter_properties, mntb_roi)
        roi_reviewed = True
        refresh_live_filter()


    def invalidate_mntb_roi(*_):
        global roi_reviewed
        roi_reviewed = False
        density_label.value = "ROI changed: accept reviewed ROI to recalculate"
        roi_volume_label.value = "ROI volume: pending review"

    roi_review_button.changed.connect(accept_mntb_roi)
    # Disconnect callbacks when this cell is rerun in an existing viewer.
    old_roi_callback = mntb_roi_layer.metadata.get("density_callback")
    if old_roi_callback is not None:
        mntb_roi_layer.events.data.disconnect(old_roi_callback)
        if hasattr(mntb_roi_layer.events, "paint"):
            mntb_roi_layer.events.paint.disconnect(old_roi_callback)
    mntb_roi_layer.events.data.connect(invalidate_mntb_roi)
    if hasattr(mntb_roi_layer.events, "paint"):
        mntb_roi_layer.events.paint.connect(invalidate_mntb_roi)
    mntb_roi_layer.metadata["density_callback"] = invalidate_mntb_roi

    refresh_live_filter()


## 11. Review the current filtered population

Rerun this cell after changing the Napari filters. It compares all raw masks with the currently retained population.

In [ ]:
if RUN_MODE != "density_only":
    if current_keep is None:
        raise RuntimeError("Run the dynamic live filtering cell first.")

    retained_properties = filter_properties.loc[current_keep].copy()
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    common_bins = np.histogram_bin_edges(
        filter_properties["volume_um3"].dropna(),
        bins=40,
    )
    axes[0].hist(
        filter_properties["volume_um3"].dropna(),
        bins=common_bins,
        color="gray",
        edgecolor="black",
        alpha=0.40,
        label=f"All masks (n={len(filter_properties)})",
    )
    axes[0].hist(
        retained_properties["volume_um3"].dropna(),
        bins=common_bins,
        color="royalblue",
        edgecolor="navy",
        alpha=0.70,
        label=f"Retained (n={len(retained_properties)})",
    )
    axes[0].set_xlabel("Mask volume (µm³)")
    axes[0].set_ylabel("Number of masks")
    axes[0].set_title("Volume before and after filtering")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    excluded = ~current_keep
    axes[1].scatter(
        filter_properties.loc[excluded, "volume_um3"],
        filter_properties.loc[excluded, "sphericity"],
        s=13,
        alpha=0.35,
        color="gray",
        label="Excluded",
    )
    axes[1].scatter(
        filter_properties.loc[current_keep, "volume_um3"],
        filter_properties.loc[current_keep, "sphericity"],
        s=16,
        alpha=0.60,
        color="royalblue",
        label="Retained",
    )
    axes[1].set_xlabel("Mask volume (µm³)")
    axes[1].set_ylabel("Sphericity")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title("Current filter classification")
    axes[1].legend()
    axes[1].grid(alpha=0.2)

    figure.tight_layout()
    plt.show()

## 12. Export the complete analysis package

This cell uses the values currently visible in the Napari panel. It saves a new self-contained result folder and does not overwrite the original microscopy TIFF.

The fixed, generic filenames avoid excessively long Windows paths when many marker channels are configured.

In [ ]:
if RUN_MODE != "density_only":
    if not roi_reviewed:
        raise RuntimeError("Review the MNTB ROI across Z and click Accept reviewed MNTB ROI before export.")
    if current_keep is None or filtered_masks is None:
        raise RuntimeError("Run the live filtering cell before exporting.")
    if int(current_keep.sum()) == 0:
        raise ValueError("The current filters retain zero masks; adjust them before export.")

    parent_window = getattr(viewer.window, "_qt_window", None)
    selected_output_folder = QFileDialog.getExistingDirectory(
        parent_window,
        "Select the parent folder for the generalized segmentation result",
        "",
    )
    if not selected_output_folder:
        raise RuntimeError("No export folder was selected.")

    sample_name = safe_filename(CUSTOM_SAMPLE_NAME or DEFAULT_INPUT.stem)
    output_parent = Path(selected_output_folder)
    sample_export_directory = output_parent / f"{sample_name}_nuclei_segmentation"
    sample_export_directory.mkdir(parents=True, exist_ok=True)
    if len(str(sample_export_directory)) >= 220:
        raise ValueError(
            "The output path is too long for a safe Windows workflow. "
            "Choose a folder closer to the drive root or shorten CUSTOM_SAMPLE_NAME."
        )

    settings = current_filter_settings()
    final_keep = apply_filter_settings(settings)
    if not final_keep.equals(current_keep):
        raise RuntimeError(
            "The displayed filter state changed. Refresh the panel and rerun export."
        )

    kept_labels = filter_properties.index[final_keep].to_numpy()
    export_masks = np.where(np.isin(masks, kept_labels), masks, 0)
    combined_perinuclear_keep = calculate_perinuclear_filter_keep(settings)
    combined_perinuclear_labels = filter_properties.index[
        combined_perinuclear_keep
    ].to_numpy()
    combined_perinuclear_masks = np.where(
        np.isin(masks, combined_perinuclear_labels),
        masks,
        0,
    )

    raw_mask_path = sample_export_directory / "cellpose_raw_masks.tif"
    combined_marker_mask_path = (
        sample_export_directory / "perinuclear_marker_associated_dapi_nuclei.tif"
    )
    filtered_mask_path = sample_export_directory / "filtered_masks_syglass.tif"
    all_masks_csv_path = sample_export_directory / "all_mask_properties.csv"
    retained_csv_path = sample_export_directory / "retained_mask_properties.csv"
    excluded_csv_path = sample_export_directory / "excluded_mask_properties.csv"
    guard_csv_path = sample_export_directory / "z_guard_excluded_mask_properties.csv"
    perinuclear_csv_path = sample_export_directory / "perinuclear_marker_metrics.csv"
    background_csv_path = sample_export_directory / "channel_background_estimates.csv"
    filters_csv_path = sample_export_directory / "filter_settings.csv"
    summary_csv_path = sample_export_directory / "analysis_summary.csv"
    summary_txt_path = sample_export_directory / "analysis_summary.txt"
    config_json_path = sample_export_directory / "analysis_config.json"
    volume_figure_path = sample_export_directory / "qc_volume_distribution.png"
    scatter_figure_path = sample_export_directory / "qc_volume_sphericity.png"
    marker_figure_path = sample_export_directory / "qc_perinuclear_markers.png"
    reason_figure_path = sample_export_directory / "qc_exclusion_reasons.png"
    report_path = sample_export_directory / "analysis_report.pdf"

    calibrated_label_tiff(raw_mask_path, masks, VOXEL_SPACING_UM, require_uint16=False)
    calibrated_label_tiff(
        combined_marker_mask_path,
        combined_perinuclear_masks,
        VOXEL_SPACING_UM,
        require_uint16=True,
    )
    calibrated_label_tiff(
        filtered_mask_path,
        export_masks,
        VOXEL_SPACING_UM,
        require_uint16=True,
    )

    individual_marker_mask_paths = []
    individual_marker_keep = {}
    for marker_name, marker_filter in settings["perinuclear_filters"].items():
        marker_keep = (
            filter_properties[marker_filter["column"]].fillna(0.0)
            >= marker_filter["minimum_positive_fraction"]
        )
        individual_marker_keep[marker_name] = marker_keep
        marker_labels = filter_properties.index[marker_keep].to_numpy()
        marker_masks = np.where(np.isin(masks, marker_labels), masks, 0)
        marker_path = sample_export_directory / (
            f"{safe_filename(marker_name)}_associated_dapi_nuclei.tif"
        )
        calibrated_label_tiff(
            marker_path,
            marker_masks,
            VOXEL_SPACING_UM,
            require_uint16=True,
        )
        individual_marker_mask_paths.append(marker_path)

    mask_results = filter_properties.copy()
    mask_results["retained"] = final_keep.astype(bool)
    mask_results["excluded_by_final_filter"] = (~final_keep).astype(bool)
    mask_results["fails_volume_filter"] = (
        mask_results["volume_um3"] < settings["minimum_volume_um3"]
    )
    mask_results["fails_sphericity_filter"] = (
        mask_results["sphericity"] < settings["minimum_sphericity"]
    ) | mask_results["sphericity"].isna()
    for marker_name, marker_keep in individual_marker_keep.items():
        mask_results[
            f"fails_perinuclear_{safe_key(marker_name).lower()}"
        ] = (~marker_keep).astype(bool)
    mask_results["fails_perinuclear_combination"] = (
        ~combined_perinuclear_keep
    ).astype(bool)

    reason_columns = {
        "excluded_by_sampling_roi": "outside sampling ROI or Z range",
        "fails_volume_filter": "physical volume",
        "fails_sphericity_filter": "sphericity",
        "fails_perinuclear_combination": "perinuclear marker association",
        "excluded_by_z_guard": "physical Z guard",
    }
    for name, intensity_filter in settings["intensity_filters"].items():
        reason_column = f"fails_intensity_filter_{safe_key(name)}"
        mask_results[reason_column] = ~mask_results[
            intensity_filter["column"]
        ].between(
            intensity_filter["minimum"],
            intensity_filter["maximum"],
            inclusive="both",
        )
        reason_columns[reason_column] = f"{name} nuclear intensity"


    def describe_exclusion(row):
        if bool(row["retained"]):
            return "retained"
        reasons = [
            description
            for column, description in reason_columns.items()
            if bool(row[column])
        ]
        return "; ".join(reasons) if reasons else "excluded by combined rule"


    mask_results["exclusion_reasons"] = mask_results.apply(describe_exclusion, axis=1)
    retained_results = mask_results.loc[mask_results["retained"]].copy()
    excluded_results = mask_results.loc[~mask_results["retained"]].copy()
    guard_results = mask_results.loc[mask_results["excluded_by_z_guard"]].copy()

    filter_rows = [
        {
            "criterion": "volume_um3",
            "channel": None,
            "minimum": settings["minimum_volume_um3"],
            "maximum": None,
            "enabled": True,
            "combination": "all",
        },
        {
            "criterion": "sphericity",
            "channel": None,
            "minimum": settings["minimum_sphericity"],
            "maximum": None,
            "enabled": True,
            "combination": "all",
        },
    ]
    for marker_name, marker_filter in settings["perinuclear_filters"].items():
        filter_rows.append(
            {
                "criterion": "perinuclear_positive_fraction",
                "channel": marker_name,
                "minimum": marker_filter["minimum_positive_fraction"],
                "maximum": 1.0,
                "enabled": True,
                "combination": settings["perinuclear_combination"],
            }
        )
    for name, intensity_filter in settings["intensity_filters"].items():
        filter_rows.append(
            {
                "criterion": "nuclear_mean_intensity",
                "channel": name,
                "minimum": intensity_filter["minimum"],
                "maximum": intensity_filter["maximum"],
                "enabled": True,
                "combination": settings["intensity_combination"],
            }
        )
    filter_table = pd.DataFrame(filter_rows)

    channel_records = []
    for item in CHANNELS:
        record = dict(item)
        if item["name"] in settings["intensity_filters"]:
            record["filter_min"] = settings["intensity_filters"][item["name"]][
                "minimum"
            ]
            record["filter_max"] = settings["intensity_filters"][item["name"]][
                "maximum"
            ]
        channel_records.append(record)

    perinuclear_records = []
    for marker_name, marker_info in PERINUCLEAR_MARKER_RESULTS.items():
        marker_filter = settings["perinuclear_filters"][marker_name]
        perinuclear_records.append(
            {
                "name": marker_name,
                "enabled": True,
                "ring_inner_um": marker_info["ring_inner_um"],
                "ring_outer_um": marker_info["ring_outer_um"],
                "configured_pixel_threshold": marker_info[
                    "configured_pixel_threshold"
                ],
                "resolved_pixel_threshold": marker_info[
                    "resolved_pixel_threshold"
                ],
                "threshold_source": marker_info["threshold_source"],
                "fraction_column": marker_info["fraction_column"],
                "minimum_positive_fraction": marker_filter[
                    "minimum_positive_fraction"
                ],
            }
        )

    configuration = {
        "schema_version": 5,
        "nominal_section_thickness_um": float(NOMINAL_SECTION_THICKNESS_UM),
        "mntb_roi": {"source": roi_source, "mask_file": "mntb_roi.tif", "reviewed": True},
        "analysis_date": datetime.now().isoformat(timespec="seconds"),
        "future_direction": "Convert this notebook into a standalone application.",
        "input": {
            "path_at_analysis": str(DEFAULT_INPUT),
            "filename": DEFAULT_INPUT.name,
            "sha256": INPUT_SHA256,
            **image_information,
            "storage": OME_ZARR_CACHE_INFO,
        },
        "channels": channel_records,
        "segmentation_channel": SEGMENTATION_CHANNEL,
        "perinuclear_markers": perinuclear_records,
        "perinuclear_combination": settings["perinuclear_combination"],
        "calibration": {
            "source": CALIBRATION_SOURCE,
            "voxel_spacing_zyx_um": list(VOXEL_SPACING_UM),
            "anisotropy_z_over_mean_xy": ANISOTROPY,
        },
        "cellpose": {
            "model_name": MODEL_NAME,
            "cellprob_threshold": CELLPROB_THRESHOLD,
            "minimum_size_voxels": CELLPOSE_MIN_SIZE_VOXELS,
            "batch_size": BATCH_SIZE,
            "flow3d_smooth": FLOW3D_SMOOTH,
            "diameter_pixels": DIAMETER_PIXELS,
            "used_gpu": bool(DEVICE.type != "cpu"),
            "device": str(DEVICE),
            "model_load_seconds": CELLPOSE_MODEL_LOAD_SECONDS,
            "evaluation_seconds": CELLPOSE_EVAL_SECONDS,
        },
        "intensity_measurements": {
            "mode": INTENSITY_MODE,
            "global_background_percentile": GLOBAL_BACKGROUND_PERCENTILE,
        },
        "filters": settings,
        "software_versions": software_versions(),
    }
    config_json_path.write_text(json.dumps(configuration, indent=2), encoding="utf-8")

    mask_results.reset_index().to_csv(all_masks_csv_path, index=False)
    retained_results.reset_index().to_csv(retained_csv_path, index=False)
    excluded_results.reset_index().to_csv(excluded_csv_path, index=False)
    guard_results.reset_index().to_csv(guard_csv_path, index=False)
    perinuclear_columns = [column for column in mask_results if "_ring_" in column]
    mask_results[perinuclear_columns].reset_index().to_csv(
        perinuclear_csv_path,
        index=False,
    )
    background_table.to_csv(background_csv_path, index=False)
    filter_table.to_csv(filters_csv_path, index=False)

    failure_counts = {
        description: int(mask_results[column].sum())
        for column, description in reason_columns.items()
    }
    summary = {
        "analysis_date": configuration["analysis_date"],
        "input_file": str(DEFAULT_INPUT),
        "sample_name": sample_name,
        "segmentation_channel": SEGMENTATION_CHANNEL,
        "perinuclear_markers": ";".join(PERINUCLEAR_MARKER_RESULTS),
        "perinuclear_combination": settings["perinuclear_combination"],
        "configured_channels": ";".join(item["name"] for item in CHANNELS),
        "intensity_mode": INTENSITY_MODE,
        "storage_backend": OME_ZARR_CACHE_INFO["storage_backend"],
        "cellpose_model_load_seconds": CELLPOSE_MODEL_LOAD_SECONDS,
        "cellpose_evaluation_seconds": CELLPOSE_EVAL_SECONDS,
        "shape_z": int(masks.shape[0]),
        "shape_y": int(masks.shape[1]),
        "shape_x": int(masks.shape[2]),
        "voxel_spacing_z_um": float(VOXEL_SPACING_UM[0]),
        "voxel_spacing_y_um": float(VOXEL_SPACING_UM[1]),
        "voxel_spacing_x_um": float(VOXEL_SPACING_UM[2]),
        "anisotropy": ANISOTROPY,
        "first_tissue_z": settings["first_tissue_z"],
        "last_tissue_z": settings["last_tissue_z"],
        "z_guard_um": settings["z_guard_um"],
        "z_guard_mode": settings["z_guard_mode"],
        "minimum_volume_um3": settings["minimum_volume_um3"],
        "minimum_sphericity": settings["minimum_sphericity"],
        "total_cellpose_masks": int(len(mask_results)),
        "combined_marker_associated_nuclei": int(combined_perinuclear_keep.sum()),
        "retained_masks": int(len(retained_results)),
        "excluded_masks": int(len(excluded_results)),
        "excluded_by_z_guard": int(len(guard_results)),
    }
    for marker_name, marker_info in PERINUCLEAR_MARKER_RESULTS.items():
        key = safe_key(marker_name).lower()
        summary[f"{key}_resolved_pixel_threshold"] = marker_info[
            "resolved_pixel_threshold"
        ]
        summary[f"{key}_minimum_positive_fraction"] = settings[
            "perinuclear_filters"
        ][marker_name]["minimum_positive_fraction"]
        summary[f"{key}_associated_nuclei"] = int(
            individual_marker_keep[marker_name].sum()
        )
    for description, count in failure_counts.items():
        summary[f"failed_{safe_key(description).lower()}"] = count
    density_summary = calculate_density_summary(
        roi_plane_counts, settings, VOXEL_SPACING_UM, int(final_keep.sum()), roi_reviewed)
    summary.update(density_summary)
    summary["roi_source"] = roi_source
    pd.DataFrame([summary]).to_csv(summary_csv_path, index=False)
    roi_export_path = sample_export_directory / "mntb_roi.tif"
    calibrated_label_tiff(roi_export_path, mntb_roi.astype(np.uint8), VOXEL_SPACING_UM)
    density_csv_path = sample_export_directory / "mntb_density_summary.csv"
    pd.DataFrame([{**density_summary, "roi_source": roi_source}]).to_csv(density_csv_path, index=False)
    _, effective_planes = density_z_planes(settings, masks.shape[0], z_spacing_um)
    roi_planes_path = sample_export_directory / "mntb_roi_by_z.csv"
    pd.DataFrame({"z_index": np.arange(masks.shape[0]), "roi_voxels": roi_plane_counts,
                  "area_um2": roi_plane_counts * VOXEL_SPACING_UM[1] * VOXEL_SPACING_UM[2],
                  "included_in_density": effective_planes}).to_csv(roi_planes_path, index=False)

    summary_lines = [
        "GENERAL CELLPPOSE-SAM 3D DAPI NUCLEI SEGMENTATION",
        "",
        f"Input: {DEFAULT_INPUT}",
        f"Segmentation channel: {SEGMENTATION_CHANNEL}",
        f"Perinuclear combination: {settings['perinuclear_combination']}",
        f"Voxel spacing ZYX: {VOXEL_SPACING_UM[0]:.4f}, {VOXEL_SPACING_UM[1]:.4f}, {VOXEL_SPACING_UM[2]:.4f} µm",
        "",
        "PERINUCLEAR MARKERS",
    ]
    for marker_record in perinuclear_records:
        summary_lines.append(
            f"{marker_record['name']}: shell {marker_record['ring_inner_um']:.2f}–"
            f"{marker_record['ring_outer_um']:.2f} µm; pixel threshold "
            f"{marker_record['resolved_pixel_threshold']:.3f}; minimum fraction "
            f"{marker_record['minimum_positive_fraction']:.3f}"
        )
    summary_lines.extend(
        [
            "",
            "CURRENT FILTERS",
            f"Minimum volume: {settings['minimum_volume_um3']:.2f} µm³",
            f"Minimum sphericity: {settings['minimum_sphericity']:.3f}",
            f"Tissue Z range: {settings['first_tissue_z']} to {settings['last_tissue_z']}",
            f"Z guard: {settings['z_guard_mode']} at {settings['z_guard_um']:.2f} µm",
            f"Nuclear intensity mode: {INTENSITY_MODE}",
            "",
            "RESULTS",
            f"Total Cellpose masks: {len(mask_results)}",
            f"Combined marker-associated nuclei: {int(combined_perinuclear_keep.sum())}",
            f"Retained masks: {len(retained_results)}",
            f"Excluded masks: {len(excluded_results)}",
            f"Excluded by Z guard: {len(guard_results)}",
        ]
    )
    summary_txt_path.write_text("\n".join(summary_lines), encoding="utf-8")

    all_volumes = mask_results["volume_um3"].dropna()
    retained_volumes = retained_results["volume_um3"].dropna()
    volume_bins = np.histogram_bin_edges(all_volumes, bins=40)
    volume_figure, volume_axis = plt.subplots(figsize=(9, 5.5))
    volume_axis.hist(
        all_volumes,
        bins=volume_bins,
        color="gray",
        edgecolor="black",
        alpha=0.40,
        label=f"All masks (n={len(all_volumes)})",
    )
    volume_axis.hist(
        retained_volumes,
        bins=volume_bins,
        color="royalblue",
        edgecolor="navy",
        alpha=0.70,
        label=f"Retained (n={len(retained_volumes)})",
    )
    volume_axis.set_xlabel("Mask volume (µm³)")
    volume_axis.set_ylabel("Number of masks")
    volume_axis.set_title("Mask volumes before and after filtering")
    volume_axis.legend()
    volume_axis.grid(alpha=0.2)
    volume_figure.tight_layout()
    volume_figure.savefig(volume_figure_path, dpi=300, bbox_inches="tight")

    scatter_figure, scatter_axis = plt.subplots(figsize=(9, 5.5))
    scatter_axis.scatter(
        excluded_results["volume_um3"],
        excluded_results["sphericity"],
        s=14,
        alpha=0.35,
        color="gray",
        label="Excluded",
    )
    scatter_axis.scatter(
        retained_results["volume_um3"],
        retained_results["sphericity"],
        s=18,
        alpha=0.65,
        color="royalblue",
        label="Retained",
    )
    scatter_axis.set_xlabel("Mask volume (µm³)")
    scatter_axis.set_ylabel("Sphericity")
    scatter_axis.set_ylim(0, 1.05)
    scatter_axis.set_title("Retained and excluded mask morphology")
    scatter_axis.legend()
    scatter_axis.grid(alpha=0.2)
    scatter_figure.tight_layout()
    scatter_figure.savefig(scatter_figure_path, dpi=300, bbox_inches="tight")

    marker_figure, marker_axis = plt.subplots(figsize=(9, 5.5))
    for marker_name, marker_filter in settings["perinuclear_filters"].items():
        marker_axis.hist(
            mask_results[marker_filter["column"]].dropna(),
            bins=np.linspace(0, 1, 41),
            alpha=0.45,
            label=marker_name,
        )
        marker_axis.axvline(
            marker_filter["minimum_positive_fraction"],
            linestyle="--",
            linewidth=1.5,
        )
    marker_axis.set_xlabel("Marker-positive perinuclear-shell fraction")
    marker_axis.set_ylabel("Number of DAPI nuclei")
    marker_axis.set_title("Perinuclear marker association")
    marker_axis.set_xlim(0, 1)
    marker_axis.legend()
    marker_axis.grid(alpha=0.2)
    marker_figure.tight_layout()
    marker_figure.savefig(marker_figure_path, dpi=300, bbox_inches="tight")

    reason_figure, reason_axis = plt.subplots(figsize=(9, 5.5))
    labels = list(failure_counts)
    counts = [failure_counts[label] for label in labels]
    reason_axis.barh(labels, counts, color="slateblue", edgecolor="navy")
    reason_axis.invert_yaxis()
    reason_axis.set_xlabel("Number of masks")
    reason_axis.set_title("Masks failing each exclusion criterion")
    reason_axis.grid(axis="x", alpha=0.2)
    reason_figure.tight_layout()
    reason_figure.savefig(reason_figure_path, dpi=300, bbox_inches="tight")

    with PdfPages(report_path) as pdf:
        metadata = pdf.infodict()
        metadata["Title"] = "General Cellpose-SAM DAPI nuclei report"
        metadata["Author"] = PDF_AUTHOR
        metadata["Subject"] = "3D nuclei and configurable perinuclear markers"
        text_figure = plt.figure(figsize=(8.27, 11.69))
        text_axis = text_figure.add_subplot(111)
        text_axis.axis("off")
        wrapped_text = "\n".join(
            textwrap.fill(line, width=95) if len(line) > 95 else line
            for line in summary_lines
        )
        text_axis.text(
            0.03,
            0.98,
            wrapped_text,
            transform=text_axis.transAxes,
            va="top",
            ha="left",
            fontsize=9.5,
            family="monospace",
        )
        pdf.savefig(text_figure, bbox_inches="tight")
        plt.close(text_figure)
        pdf.savefig(volume_figure, bbox_inches="tight")
        pdf.savefig(scatter_figure, bbox_inches="tight")
        pdf.savefig(marker_figure, bbox_inches="tight")
        pdf.savefig(reason_figure, bbox_inches="tight")

    plt.close(volume_figure)
    plt.close(scatter_figure)
    plt.close(marker_figure)
    plt.close(reason_figure)

    density_txt_path = sample_export_directory / "mntb_density_summary.txt"
    density_txt_path.write_text("\n".join(f"{k}: {v}" for k, v in density_summary.items()), encoding="utf-8")
    created_files = [
        roi_export_path, density_csv_path, roi_planes_path, density_txt_path,
        raw_mask_path,
        *individual_marker_mask_paths,
        combined_marker_mask_path,
        filtered_mask_path,
        all_masks_csv_path,
        retained_csv_path,
        excluded_csv_path,
        guard_csv_path,
        perinuclear_csv_path,
        background_csv_path,
        filters_csv_path,
        summary_csv_path,
        summary_txt_path,
        config_json_path,
        volume_figure_path,
        scatter_figure_path,
        marker_figure_path,
        reason_figure_path,
        report_path,
    ]

    print(f"Analysis package saved to:\n{sample_export_directory}")
    print("\nCreated files:")
    for created_file in created_files:
        print(f"  {created_file.name}")


## Notes for the future app conversion

The app should preserve the same boundaries used here:

1. **Configuration/UI layer:** file selection, channel table, Cellpose settings, and filter controls.
2. **Analysis core:** TIFF standardization, calibration, segmentation, measurements, guard calculation, and filtering.
3. **Visualization layer:** Napari or an equivalent embedded viewer.
4. **Export layer:** calibrated masks, CSV tables, JSON settings, QC figures, and PDF report.

Keeping these layers separate will allow the notebook functions to become importable Python modules and the interactive controls to become an application interface.
5. **Storage layer:** keep OME-Zarr optional. The app should explain that it improves caching and large-image access, while tiling/out-of-core inference is a separate future feature required to change Cellpose processing time materially.
